# Notebook 5 — Future Customer Value

## Objective

This notebook develops a forward-looking model of customer economic value.

The objective is to estimate the amount of merchandise revenue that each identified customer is expected to generate during the next 90 days.

The primary target is therefore:

> **90-Day Future Customer Value (FCV90)**

defined as the customer's positive net merchandise revenue generated during the 90 days following the prediction date.

This is not a true lifetime-value model.

The available retail dataset observes only a finite historical period, so the notebook deliberately estimates a fixed-horizon future value rather than claiming to model the customer's entire economic lifetime.

The business question is:

> **Which customers are expected to create the most revenue during the next 90 days?**

This extends the earlier customer analytics:

- reactivation modelling identifies customers likely to return;
- revenue-leakage modelling identifies customers at risk of deterioration;
- Future Customer Value estimates the expected economic contribution of each customer over a defined forward horizon.

## Why Use a 90-Day Future Value Horizon?

A fixed 90-day horizon provides several advantages.

It:

- matches the forward-looking horizon used in the earlier customer models;
- creates a directly observable target;
- avoids unsupported assumptions about permanent customer lifetime;
- supports chronological model validation;
- and creates an economically interpretable near-term planning measure.

A customer's FCV90 may equal zero if they generate no qualifying merchandise revenue during the following 90 days.

This zero outcome is economically meaningful and should not be treated as missing data.

## Initial Modelling Hypothesis

Future customer revenue is expected to have two important statistical characteristics:

1. many customers may generate **zero revenue** during the future 90-day window;
2. positive future revenue is likely to be strongly right-skewed.

For that reason, the notebook will first inspect the target distribution before selecting the final modelling architecture.

A likely candidate is a two-part or hurdle framework:

### Part 1 — Future Activity Probability

Estimate:

> **P(Future 90-Day Revenue > 0)**

### Part 2 — Conditional Future Revenue

For customers generating positive future revenue, estimate:

> **Expected Future Revenue | Future Revenue > 0**

### Combined Expected Future Customer Value

The two components can then be combined as:

> **Expected FCV90 = Probability of Future Activity × Expected Positive Future Revenue**

This architecture will only be adopted after the target diagnostics confirm that it is appropriate.

In [ ]:
# ================================================================
# 47.4 — IMPORTS
# ================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ================================================================
# 47.5 — LOAD CANONICAL CLEANED DATA
# ================================================================

df = pd.read_parquet(
    "cleaned_online_retail.parquet"
)


print(
    f"Rows loaded: {len(df):,}"
)

print(
    f"Columns loaded: {df.shape[1]:,}"
)

In [ ]:
# ================================================================
# 47.6 — REQUIRED COLUMN VALIDATION
# ================================================================

required_columns = [
    "date",
    "invoice",
    "customer",
    "product",
    "region",
    "quantity",
    "unit_price",
    "customer_is_identified",
    "is_merchandise",
    "is_cancellation",
    "net_merchandise_revenue",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)


print(
    "✓ Required columns confirmed."
)

In [ ]:
# ================================================================
# 47.7 — MODELLING OBSERVATION END
# ================================================================

MODELLING_END_DATE = pd.Timestamp(
    "2011-11-30 23:59:59"
)


print(
    f"Modelling observation boundary: "
    f"{MODELLING_END_DATE}"
)

In [ ]:
# ================================================================
# 47.8 — QUALIFYING POSITIVE CUSTOMER PURCHASES
# ================================================================

positive_customer_purchases = (
    df.loc[
        df[
            "customer_is_identified"
        ]
        &
        df[
            "is_merchandise"
        ]
        &
        (
            df[
                "quantity"
            ]
            > 0
        )
        &
        (
            df[
                "unit_price"
            ]
            > 0
        )
        &
        (
            ~df[
                "is_cancellation"
            ]
        )
        &
        (
            df[
                "date"
            ]
            <=
            MODELLING_END_DATE
        )
    ]
    .copy()
)


positive_customer_purchases[
    "purchase_day"
] = (
    positive_customer_purchases[
        "date"
    ]
    .dt.normalize()
)


print(
    f"Qualifying rows: "
    f"{len(positive_customer_purchases):,}"
)

print(
    f"Identified customers: "
    f"{positive_customer_purchases['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 47.9 — CUSTOMER-DAY AGGREGATION
# ================================================================

customer_daily = (
    positive_customer_purchases
    .groupby(
        [
            "customer",
            pd.Grouper(
                key="date",
                freq="D"
            )
        ],
        as_index=False
    )
    .agg(
        daily_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        daily_units=(
            "quantity",
            "sum"
        ),

        daily_orders=(
            "invoice",
            "nunique"
        ),

        daily_products=(
            "product",
            "nunique"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


customer_daily[
    "date"
] = (
    pd.to_datetime(
        customer_daily[
            "date"
        ]
    )
    .dt.normalize()
)


print(
    f"Customer-day rows: "
    f"{len(customer_daily):,}"
)

## Prediction Population

A customer is eligible at a monthly prediction date if they generated at least one qualifying merchandise purchase during the preceding **180 days**.

The customer does not need to have purchased during the most recent 90 days.

This is intentional.

A future-value model should be able to score:

- currently active customers;
- recently inactive customers;
- and customers whose purchasing activity is beginning to weaken.

The 180-day eligibility window prevents extremely stale historical relationships from remaining indefinitely in the scoring population.

In [ ]:
# ================================================================
# 47.11 — MONTH-END PREDICTION DATES
# ================================================================

PREDICTION_START_DATE = pd.Timestamp(
    "2010-05-31"
)


PREDICTION_END_DATE = pd.Timestamp(
    "2011-08-31"
)


prediction_dates = pd.date_range(
    start=PREDICTION_START_DATE,
    end=PREDICTION_END_DATE,
    freq="ME"
)


print(
    f"Prediction dates: "
    f"{len(prediction_dates)}"
)

print(
    prediction_dates
)

In [ ]:
# ================================================================
# 47.12 — WINDOW DEFINITIONS
# ================================================================

ELIGIBILITY_LOOKBACK_DAYS = 180
RECENT_WINDOW_DAYS = 90
OUTCOME_WINDOW_DAYS = 90


print(
    f"Eligibility/history window: "
    f"{ELIGIBILITY_LOOKBACK_DAYS} days"
)

print(
    f"Recent window: "
    f"{RECENT_WINDOW_DAYS} days"
)

print(
    f"Future value horizon: "
    f"{OUTCOME_WINDOW_DAYS} days"
)

### 47.13 — Build candidate FCV90 episodes
Each row will represent:
> one customer at one prediction date.

In [ ]:
# ================================================================
# 47.13 — BUILD FCV90 PREDICTION EPISODES
# ================================================================

fcv_episode_records = []


for prediction_date in prediction_dates:

    history_start = (
        prediction_date
        -
        pd.Timedelta(
            days=ELIGIBILITY_LOOKBACK_DAYS
        )
    )


    outcome_end = (
        prediction_date
        +
        pd.Timedelta(
            days=OUTCOME_WINDOW_DAYS
        )
    )


    # ------------------------------------------------------------
    # Eligible customers:
    # at least one purchase during previous 180 days
    # ------------------------------------------------------------

    history_window = (
        customer_daily.loc[
            (
                customer_daily[
                    "date"
                ]
                >
                history_start
            )
            &
            (
                customer_daily[
                    "date"
                ]
                <=
                prediction_date
            )
        ]
        .copy()
    )


    eligible_customers = (
        history_window[
            "customer"
        ]
        .drop_duplicates()
    )


    eligibility_summary = (
        history_window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            history_180d_revenue=(
                "daily_revenue",
                "sum"
            ),

            history_180d_orders=(
                "daily_orders",
                "sum"
            ),

            history_180d_active_days=(
                "date",
                "nunique"
            ),

            history_180d_units=(
                "daily_units",
                "sum"
            ),

            history_180d_first_purchase=(
                "date",
                "min"
            ),

            history_180d_last_purchase=(
                "date",
                "max"
            ),

            last_region=(
                "region",
                "last"
            ),
        )
    )


    # ------------------------------------------------------------
    # Future 90-day revenue
    # ------------------------------------------------------------

    future_window = (
        customer_daily.loc[
            (
                customer_daily[
                    "date"
                ]
                >
                prediction_date
            )
            &
            (
                customer_daily[
                    "date"
                ]
                <=
                outcome_end
            )
            &
            (
                customer_daily[
                    "customer"
                ]
                .isin(
                    eligible_customers
                )
            )
        ]
        .copy()
    )


    future_summary = (
        future_window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            future_90d_revenue=(
                "daily_revenue",
                "sum"
            ),

            future_90d_orders=(
                "daily_orders",
                "sum"
            ),

            future_90d_active_days=(
                "date",
                "nunique"
            ),

            future_90d_units=(
                "daily_units",
                "sum"
            ),
        )
    )


    # ------------------------------------------------------------
    # Merge
    # ------------------------------------------------------------

    episode = (
        eligibility_summary
        .merge(
            future_summary,
            on="customer",
            how="left",
            validate="one_to_one"
        )
    )


    future_zero_fill = [
        "future_90d_revenue",
        "future_90d_orders",
        "future_90d_active_days",
        "future_90d_units",
    ]


    episode[
        future_zero_fill
    ] = (
        episode[
            future_zero_fill
        ]
        .fillna(0)
    )


    episode[
        "prediction_date"
    ] = prediction_date


    episode[
        "history_start"
    ] = history_start


    episode[
        "outcome_end"
    ] = outcome_end


    fcv_episode_records.append(
        episode
    )

In [ ]:
fcv90_episodes = pd.concat(
    fcv_episode_records,
    ignore_index=True
)


print(
    f"FCV90 prediction episodes: "
    f"{len(fcv90_episodes):,}"
)

print(
    f"Unique customers: "
    f"{fcv90_episodes['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 47.14 — PRIMARY TARGET
# ================================================================

fcv90_episodes[
    "fcv90"
] = (
    fcv90_episodes[
        "future_90d_revenue"
    ]
)


fcv90_episodes[
    "future_active"
] = (
    fcv90_episodes[
        "fcv90"
    ]
    > 0
).astype(int)


print(
    "✓ FCV90 target created."
)

In [ ]:
# ================================================================
# 47.15 — FCV90 TARGET SUMMARY
# ================================================================

fcv90_target_summary = pd.DataFrame(
    {
        "Metric": [
            "Prediction episodes",
            "Unique customers",
            "Future active episodes",
            "Future zero-revenue episodes",
            "Future activity rate",
            "Mean FCV90",
            "Median FCV90",
            "Total episode-level future revenue",
        ],

        "Value": [
            len(
                fcv90_episodes
            ),

            fcv90_episodes[
                "customer"
            ].nunique(),

            fcv90_episodes[
                "future_active"
            ].sum(),

            (
                fcv90_episodes[
                    "future_active"
                ]
                == 0
            ).sum(),

            fcv90_episodes[
                "future_active"
            ].mean(),

            fcv90_episodes[
                "fcv90"
            ].mean(),

            fcv90_episodes[
                "fcv90"
            ].median(),

            fcv90_episodes[
                "fcv90"
            ].sum(),
        ],
    }
)


display(
    fcv90_target_summary
)

In [ ]:
# ================================================================
# 47.16 — FCV90 DISTRIBUTION
# ================================================================

fcv90_distribution_summary = (
    fcv90_episodes[
        "fcv90"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


display(
    fcv90_distribution_summary
)

In [ ]:
# ================================================================
# 47.17 — POSITIVE FCV90 DISTRIBUTION
# ================================================================

positive_fcv90 = (
    fcv90_episodes.loc[
        fcv90_episodes[
            "fcv90"
        ]
        > 0,
        "fcv90"
    ]
)


positive_fcv90_summary = (
    positive_fcv90
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


display(
    positive_fcv90_summary
)

In [ ]:
# ================================================================
# 47.18 — ZERO / POSITIVE TARGET MIX
# ================================================================

fcv90_zero_positive_summary = (
    fcv90_episodes[
        "future_active"
    ]
    .value_counts()
    .rename_axis(
        "future_active"
    )
    .reset_index(
        name="episodes"
    )
)


fcv90_zero_positive_summary[
    "share_pct"
] = (
    fcv90_zero_positive_summary[
        "episodes"
    ]
    /
    len(
        fcv90_episodes
    )
    *
    100
)


display(
    fcv90_zero_positive_summary
)

In [ ]:
# ================================================================
# VISUAL 47.1 — FCV90 DISTRIBUTION
# ================================================================

fcv90_visual_cap = (
    fcv90_episodes[
        "fcv90"
    ]
    .quantile(
        0.99
    )
)


plt.figure(
    figsize=(11, 6)
)


sns.histplot(
    fcv90_episodes[
        "fcv90"
    ]
    .clip(
        upper=fcv90_visual_cap
    ),
    bins=60
)


plt.title(
    "Future 90-Day Customer Revenue Distribution"
)

plt.xlabel(
    "FCV90 — visual capped at 99th percentile"
)

plt.ylabel(
    "Prediction Episodes"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 47.2 — POSITIVE FCV90 LOG DISTRIBUTION
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.histplot(
    np.log1p(
        positive_fcv90
    ),
    bins=60
)


plt.title(
    "Positive Future 90-Day Customer Revenue — Log Scale"
)

plt.xlabel(
    "log(1 + FCV90)"
)

plt.ylabel(
    "Prediction Episodes"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 47.19 — MONTHLY FCV90 SUMMARY
# ================================================================

fcv90_episodes[
    "prediction_period"
] = (
    fcv90_episodes[
        "prediction_date"
    ]
    .dt.to_period(
        "M"
    )
)


monthly_fcv90_summary = (
    fcv90_episodes
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        unique_customers=(
            "customer",
            "nunique"
        ),

        future_activity_rate=(
            "future_active",
            "mean"
        ),

        average_fcv90=(
            "fcv90",
            "mean"
        ),

        median_fcv90=(
            "fcv90",
            "median"
        ),

        total_fcv90=(
            "fcv90",
            "sum"
        ),
    )
)


monthly_fcv90_summary[
    "future_activity_rate_pct"
] = (
    monthly_fcv90_summary[
        "future_activity_rate"
    ]
    *
    100
)


display(
    monthly_fcv90_summary
)

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=monthly_fcv90_summary,
    x=monthly_fcv90_summary[
        "prediction_period"
    ].astype(str),
    y="future_activity_rate_pct",
    marker="o"
)


plt.title(
    "Future 90-Day Activity Rate by Prediction Month"
)

plt.xlabel(
    "Prediction Month"
)

plt.ylabel(
    "Future Activity Rate (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=monthly_fcv90_summary,
    x=monthly_fcv90_summary[
        "prediction_period"
    ].astype(str),
    y="median_fcv90",
    marker="o"
)


plt.title(
    "Median Future 90-Day Customer Revenue by Prediction Month"
)

plt.xlabel(
    "Prediction Month"
)

plt.ylabel(
    "Median FCV90"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

### 47.20 — Concentration of future customer value
This is critical.

We need to know how much future revenue is concentrated among the highest-value actual outcomes.

In [ ]:
# ================================================================
# 47.20 — ACTUAL FUTURE VALUE CONCENTRATION
# ================================================================

fcv90_ranked_actual = (
    fcv90_episodes
    .sort_values(
        "fcv90",
        ascending=False
    )
    .reset_index(drop=True)
)


total_fcv90 = (
    fcv90_ranked_actual[
        "fcv90"
    ]
    .sum()
)


future_value_concentration_records = []


for fraction in [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
]:

    n_top = int(
        np.ceil(
            len(
                fcv90_ranked_actual
            )
            *
            fraction
        )
    )


    captured_value = (
        fcv90_ranked_actual
        .head(
            n_top
        )[
            "fcv90"
        ]
        .sum()
    )


    future_value_concentration_records.append(
        {
            "Episode Fraction":
                fraction,

            "Episodes":
                n_top,

            "Future Revenue Captured":
                captured_value,

            "Share of Future Revenue":
                (
                    captured_value
                    /
                    total_fcv90
                ),

            "Share of Future Revenue (%)":
                (
                    captured_value
                    /
                    total_fcv90
                    *
                    100
                ),
        }
    )


future_value_concentration = pd.DataFrame(
    future_value_concentration_records
)


display(
    future_value_concentration
)

### 47.21 — Historical revenue vs future revenue
Before ML, establish how much simple recent revenue explains.

In [ ]:
# ================================================================
# 47.21 — HISTORY VS FUTURE VALUE
# ================================================================

history_future_value_relationship = (
    fcv90_episodes[
        [
            "history_180d_revenue",
            "history_180d_orders",
            "history_180d_active_days",
            "fcv90",
        ]
    ]
    .corr(
        method="spearman"
    )
)


display(
    history_future_value_relationship
)

In [ ]:
# ================================================================
# 47.22 — RECENCY FEATURE FOR INITIAL DIAGNOSTICS
# ================================================================

fcv90_episodes[
    "days_since_last_purchase"
] = (
    fcv90_episodes[
        "prediction_date"
    ]
    -
    fcv90_episodes[
        "history_180d_last_purchase"
    ]
).dt.days

In [ ]:
# ================================================================
# 47.23 — FUTURE VALUE BY RECENCY
# ================================================================

fcv90_episodes[
    "recency_band"
] = pd.cut(
    fcv90_episodes[
        "days_since_last_purchase"
    ],
    bins=[
        -1,
        30,
        60,
        90,
        120,
        150,
        180,
    ],
    labels=[
        "0–30",
        "31–60",
        "61–90",
        "91–120",
        "121–150",
        "151–180",
    ]
)


fcv90_by_recency = (
    fcv90_episodes
    .groupby(
        "recency_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        future_activity_rate=(
            "future_active",
            "mean"
        ),

        average_fcv90=(
            "fcv90",
            "mean"
        ),

        median_fcv90=(
            "fcv90",
            "median"
        ),
    )
)


fcv90_by_recency[
    "future_activity_rate_pct"
] = (
    fcv90_by_recency[
        "future_activity_rate"
    ]
    *
    100
)


display(
    fcv90_by_recency
)

In [ ]:
# ================================================================
# 47.24 — FCV90 BY HISTORICAL REVENUE QUINTILE
# ================================================================

fcv90_episodes[
    "historical_revenue_band"
] = pd.qcut(
    fcv90_episodes[
        "history_180d_revenue"
    ],
    q=5,
    labels=[
        "Q1 — Lowest",
        "Q2",
        "Q3",
        "Q4",
        "Q5 — Highest",
    ],
    duplicates="drop"
)


fcv90_by_history_band = (
    fcv90_episodes
    .groupby(
        "historical_revenue_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        median_historical_revenue=(
            "history_180d_revenue",
            "median"
        ),

        future_activity_rate=(
            "future_active",
            "mean"
        ),

        average_fcv90=(
            "fcv90",
            "mean"
        ),

        median_fcv90=(
            "fcv90",
            "median"
        ),
    )
)


fcv90_by_history_band[
    "future_activity_rate_pct"
] = (
    fcv90_by_history_band[
        "future_activity_rate"
    ]
    *
    100
)


display(
    fcv90_by_history_band
)

In [ ]:
# ================================================================
# 47.25 — REPEATED PREDICTION EPISODES
# ================================================================

fcv90_episodes_per_customer = (
    fcv90_episodes
    .groupby(
        "customer"
    )
    .size()
)


fcv90_episode_frequency_summary = (
    fcv90_episodes_per_customer
    .describe()
)


display(
    fcv90_episode_frequency_summary
)


print(
    "Customers with multiple prediction episodes: "
    f"{(fcv90_episodes_per_customer > 1).sum():,}"
)

print(
    "Maximum episodes for one customer: "
    f"{fcv90_episodes_per_customer.max():,}"
)

## Target Architecture Decision Rule

The next modelling step will be selected from the observed FCV90 distribution.

A single regression model may be reasonable if future revenue is sufficiently continuous and the zero mass is limited.

A two-part model will be preferred if the target shows both:

- a substantial zero-revenue population;
- and a strongly skewed positive-revenue distribution.

Under the two-part architecture:

### Stage A

Classification model:

> **Will the customer generate any qualifying revenue during the next 90 days?**

### Stage B

Conditional regression model:

> **If the customer is active, how much revenue will they generate?**

### Final Estimate

> **Expected FCV90 = P(Future Active) × Predicted Positive Revenue**

This structure prevents the regression model from being forced to explain both inactivity and positive spending magnitude using one target-generating process.

### Interpretation of Episode-Level Revenue

Monthly customer prediction windows overlap.

A customer may therefore contribute to several prediction episodes, and future 90-day revenue windows may overlap through time.

For this reason:

> **the sum of FCV90 across all historical prediction episodes is not unique company revenue**

It is a modelling quantity used to evaluate customer-value prediction.

Financial-statement revenue reconciliation should continue to use non-overlapping accounting periods.

In [ ]:
# ================================================================
# 47.28 — STEP 47 COMPLETION CHECK
# ================================================================

assert (
    len(
        fcv90_episodes
    )
    > 0
)


assert (
    fcv90_episodes[
        "fcv90"
    ]
    .ge(0)
    .all()
)


assert (
    fcv90_episodes[
        "outcome_end"
    ]
    .max()
    <=
    MODELLING_END_DATE
)


assert (
    fcv90_episodes[
        "history_180d_revenue"
    ]
    .gt(0)
    .all()
)


print(
    "✓ Step 47 complete."
)

print(
    "✓ 180-day recent-customer eligibility defined."
)

print(
    "✓ 90-day Future Customer Value target created."
)

print(
    "✓ Zero and positive future-value outcomes separated."
)

print(
    "✓ Future-value concentration prepared."
)

print(
    "✓ No modelling architecture has yet been prematurely locked."
)

### Step 47 Summary — Future Customer Value Target Architecture

Step 47 converted future customer economics into a fixed-horizon prediction problem.

The primary target is:

> **FCV90 — Future 90-Day Customer Value**

defined as qualifying customer merchandise revenue generated during the 90 days following each prediction date.

This is deliberately a fixed-horizon future-value target rather than a claim about true lifetime customer value.

### The Target Has a Clear Two-Part Structure

Across **47,933 prediction episodes**, approximately:

- **49.1%** generated positive future revenue;
- **50.9%** generated zero future revenue.

The overall median FCV90 was therefore **zero**.

Future customer revenue is not generated by one continuous process.

A customer must first become economically active during the future window before a positive revenue amount can occur.

### Positive Future Revenue Is Strongly Right-Skewed

Among positive FCV90 observations:

- median future revenue was approximately **503**;
- mean future revenue was approximately **1,160**;
- the 95th percentile was approximately **3,112**;
- the 99th percentile exceeded **12,113**;
- and the maximum exceeded **117,000**.

A small number of very high-value outcomes therefore create a substantial positive tail.

### Future Value Is Highly Concentrated

Future revenue was exceptionally concentrated.

Approximately:

- the highest **1%** of actual FCV90 episodes represented **33.7%** of future revenue;
- the highest **5%** represented **55.3%**;
- the highest **10%** represented **68.4%**;
- the highest **20%** represented **83.7%**.

Future-value modelling must therefore evaluate economic ranking and revenue capture in addition to ordinary regression error.

### Recency Strongly Predicts Future Value

Customers who purchased during the previous 30 days generated substantially greater future economic value than customers whose last purchase occurred much earlier.

Future activity declined from approximately **64.5%** among customers with 0–30 day recency to approximately **27.0%** among customers with 151–180 day recency.

Average FCV90 fell from approximately **1,099** to approximately **171** across the same range.

### Historical Customer Value Strongly Predicts Future Value

Historical revenue also showed a pronounced relationship with future economics.

The highest historical-revenue quintile achieved approximately:

- **78.1% future activity**
- and **1,958 average FCV90**

compared with approximately:

- **27.0% future activity**
- and **105 average FCV90**

for the lowest quintile.

Historical scale is therefore expected to be a major predictor of future customer value.

## Modelling Architecture Decision

The primary modelling architecture will be a:

> **Two-Part Hurdle Model**

### Stage A — Future Activity Model

Predict:

> **P(FCV90 > 0)**

### Stage B — Positive Future Value Model

Among positive future-revenue episodes, predict:

> **E(FCV90 | FCV90 > 0)**

Because positive future revenue is strongly right-skewed, the initial regression target will use:

> **log(1 + FCV90)**

### Final Expected Future Customer Value

The two components will be combined as:

> **Expected FCV90 = P(Future Active) × Expected Positive FCV90**

This framework separates future purchasing probability from future spending magnitude and is therefore better aligned with the observed target-generating process than a single-stage regression.

## Step 48 — Leakage-Safe Feature Engineering and Temporal Split

Step 48 constructs the predictive feature table for the Future Customer Value hurdle model.

All predictors must be observable on or before the prediction date.

Features will describe:

- recent customer revenue;
- earlier customer revenue;
- customer purchasing frequency;
- recency;
- active purchase days;
- units;
- product breadth;
- average order economics;
- customer-history momentum;
- purchase cadence;
- historical relationship depth;
- and seasonal context.

The same underlying feature architecture will support both:

1. the Future Activity classifier;
2. the Positive FCV90 regression model.

The final Test population will remain unopened until both hurdle-model components have been selected and frozen.

In [ ]:
# ================================================================
# 48.2 — CUSTOMER WINDOW SUMMARY
# ================================================================

def summarise_fcv_customer_window(
    daily_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        daily_data.loc[
            daily_data["customer"].isin(customers)
            &
            (daily_data["date"] > start_date)
            &
            (daily_data["date"] <= end_date)
        ]
        .copy()
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            revenue=(
                "daily_revenue",
                "sum"
            ),

            orders=(
                "daily_orders",
                "sum"
            ),

            active_days=(
                "date",
                "nunique"
            ),

            units=(
                "daily_units",
                "sum"
            ),

            first_purchase_day=(
                "date",
                "min"
            ),

            last_purchase_day=(
                "date",
                "max"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column: f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

In [ ]:
# ================================================================
# 48.3 — PRODUCT WINDOW SUMMARY
# ================================================================

def summarise_fcv_product_window(
    line_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        line_data.loc[
            line_data["customer"].isin(customers)
            &
            (line_data["purchase_day"] > start_date)
            &
            (line_data["purchase_day"] <= end_date)
        ]
        .copy()
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            unique_products=(
                "product",
                "nunique"
            ),

            average_unit_price=(
                "unit_price",
                "mean"
            ),

            maximum_unit_price=(
                "unit_price",
                "max"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column: f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

In [ ]:
# ================================================================
# 48.4 — PURCHASE-GAP SUMMARY
# ================================================================

def summarise_fcv_purchase_gaps(
    daily_data,
    customers,
    start_date,
    end_date,
    prefix
):

    window = (
        daily_data.loc[
            daily_data["customer"].isin(customers)
            &
            (daily_data["date"] > start_date)
            &
            (daily_data["date"] <= end_date),
            [
                "customer",
                "date",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "customer",
                "date",
            ]
        )
        .copy()
    )


    window["gap_days"] = (
        window
        .groupby("customer")["date"]
        .diff()
        .dt.days
    )


    summary = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            average_gap_days=(
                "gap_days",
                "mean"
            ),

            maximum_gap_days=(
                "gap_days",
                "max"
            ),

            minimum_gap_days=(
                "gap_days",
                "min"
            ),
        )
    )


    summary = summary.rename(
        columns={
            column: f"{prefix}_{column}"
            for column in summary.columns
            if column != "customer"
        }
    )


    return summary

### 48.5 — Build historical feature windows
We'll use:
- h180: entire preceding 180 days
- r90: most recent 90 days
- p90: preceding 90 days
- r30: most recent 30 days

In [ ]:
# ================================================================
# 48.5 — BUILD FCV FEATURE WINDOWS
# ================================================================

fcv_feature_frames = []


for prediction_date in sorted(
    fcv90_episodes[
        "prediction_date"
    ].unique()
):

    prediction_date = pd.Timestamp(
        prediction_date
    )


    episode_population = (
        fcv90_episodes.loc[
            fcv90_episodes[
                "prediction_date"
            ]
            ==
            prediction_date
        ]
        .copy()
    )


    customers = (
        episode_population[
            "customer"
        ]
        .unique()
    )


    history180_start = (
        prediction_date
        -
        pd.Timedelta(days=180)
    )


    recent90_start = (
        prediction_date
        -
        pd.Timedelta(days=90)
    )


    recent30_start = (
        prediction_date
        -
        pd.Timedelta(days=30)
    )


    # Full historical 180 days
    h180 = summarise_fcv_customer_window(
        customer_daily,
        customers,
        history180_start,
        prediction_date,
        "h180"
    )


    # Most recent 90 days
    r90 = summarise_fcv_customer_window(
        customer_daily,
        customers,
        recent90_start,
        prediction_date,
        "r90"
    )


    # Previous 90 days
    p90 = summarise_fcv_customer_window(
        customer_daily,
        customers,
        history180_start,
        recent90_start,
        "p90"
    )


    # Most recent 30 days
    r30 = summarise_fcv_customer_window(
        customer_daily,
        customers,
        recent30_start,
        prediction_date,
        "r30"
    )


    # Product breadth
    h180_products = summarise_fcv_product_window(
        positive_customer_purchases,
        customers,
        history180_start,
        prediction_date,
        "h180"
    )


    r90_products = summarise_fcv_product_window(
        positive_customer_purchases,
        customers,
        recent90_start,
        prediction_date,
        "r90"
    )


    p90_products = summarise_fcv_product_window(
        positive_customer_purchases,
        customers,
        history180_start,
        recent90_start,
        "p90"
    )


    # Historical cadence
    h180_gaps = summarise_fcv_purchase_gaps(
        customer_daily,
        customers,
        history180_start,
        prediction_date,
        "h180"
    )


    features = (
        episode_population
        .merge(
            h180,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            r90,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            p90,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            r30,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            h180_products,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            r90_products,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            p90_products,
            on="customer",
            how="left",
            validate="one_to_one"
        )
        .merge(
            h180_gaps,
            on="customer",
            how="left",
            validate="one_to_one"
        )
    )


    fcv_feature_frames.append(
        features
    )

In [ ]:
fcv_feature_table = pd.concat(
    fcv_feature_frames,
    ignore_index=True
)


print(
    f"FCV feature-table episodes: "
    f"{len(fcv_feature_table):,}"
)

### 48.6 — Structural zero filling
A customer may have no purchase in r90, p90 or r30.

That should become zero—not missing.

In [ ]:
# ================================================================
# 48.6 — STRUCTURAL ZERO FILL
# ================================================================

zero_fill_features = [
    "r90_revenue",
    "r90_orders",
    "r90_active_days",
    "r90_units",
    "r90_unique_products",

    "p90_revenue",
    "p90_orders",
    "p90_active_days",
    "p90_units",
    "p90_unique_products",

    "r30_revenue",
    "r30_orders",
    "r30_active_days",
    "r30_units",
]


for feature in zero_fill_features:

    if feature in fcv_feature_table.columns:

        fcv_feature_table[
            feature
        ] = (
            fcv_feature_table[
                feature
            ]
            .fillna(0)
        )

In [ ]:
# ================================================================
# 48.7 — HISTORICAL REVENUE RECONSTRUCTION
# ================================================================

history_revenue_difference = (
    fcv_feature_table[
        "history_180d_revenue"
    ]
    -
    fcv_feature_table[
        "h180_revenue"
    ]
)


print(
    "Maximum absolute 180-day revenue difference: "
    f"{history_revenue_difference.abs().max():.8f}"
)


assert np.allclose(
    fcv_feature_table[
        "history_180d_revenue"
    ],
    fcv_feature_table[
        "h180_revenue"
    ]
)


print(
    "✓ 180-day revenue reconstruction confirmed."
)

In [ ]:
# ================================================================
# 48.8 — RECENCY
# ================================================================

fcv_feature_table[
    "days_since_last_purchase"
] = (
    fcv_feature_table[
        "prediction_date"
    ]
    -
    fcv_feature_table[
        "h180_last_purchase_day"
    ]
).dt.days

In [ ]:
# ================================================================
# 48.9 — CURRENT ACTIVITY INDICATORS
# ================================================================

fcv_feature_table[
    "active_in_last_30d"
] = (
    fcv_feature_table[
        "r30_orders"
    ]
    > 0
).astype(int)


fcv_feature_table[
    "active_in_last_90d"
] = (
    fcv_feature_table[
        "r90_orders"
    ]
    > 0
).astype(int)

In [ ]:
# ================================================================
# 48.10 — ECONOMIC FEATURES
# ================================================================

fcv_feature_table[
    "h180_average_order_value"
] = (
    fcv_feature_table[
        "h180_revenue"
    ]
    /
    fcv_feature_table[
        "h180_orders"
    ]
)


fcv_feature_table[
    "h180_revenue_per_active_day"
] = (
    fcv_feature_table[
        "h180_revenue"
    ]
    /
    fcv_feature_table[
        "h180_active_days"
    ]
)


fcv_feature_table[
    "h180_units_per_order"
] = (
    fcv_feature_table[
        "h180_units"
    ]
    /
    fcv_feature_table[
        "h180_orders"
    ]
)


fcv_feature_table[
    "h180_products_per_order"
] = (
    fcv_feature_table[
        "h180_unique_products"
    ]
    /
    fcv_feature_table[
        "h180_orders"
    ]
)

In [ ]:
# ================================================================
# 48.11 — RECENT 90-DAY ECONOMICS
# ================================================================

fcv_feature_table[
    "r90_average_order_value"
] = np.where(
    fcv_feature_table[
        "r90_orders"
    ]
    > 0,

    fcv_feature_table[
        "r90_revenue"
    ]
    /
    fcv_feature_table[
        "r90_orders"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 48.12 — PRIOR 90-DAY ECONOMICS
# ================================================================

fcv_feature_table[
    "p90_average_order_value"
] = np.where(
    fcv_feature_table[
        "p90_orders"
    ]
    > 0,

    fcv_feature_table[
        "p90_revenue"
    ]
    /
    fcv_feature_table[
        "p90_orders"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 48.13 — REVENUE MOMENTUM
# ================================================================

fcv_feature_table[
    "r90_minus_p90_revenue"
] = (
    fcv_feature_table[
        "r90_revenue"
    ]
    -
    fcv_feature_table[
        "p90_revenue"
    ]
)


fcv_feature_table[
    "r90_to_p90_revenue_ratio"
] = np.where(
    fcv_feature_table[
        "p90_revenue"
    ]
    > 0,

    fcv_feature_table[
        "r90_revenue"
    ]
    /
    fcv_feature_table[
        "p90_revenue"
    ],

    np.nan
)


fcv_feature_table[
    "log_revenue_momentum"
] = (
    np.log1p(
        fcv_feature_table[
            "r90_revenue"
        ]
    )
    -
    np.log1p(
        fcv_feature_table[
            "p90_revenue"
        ]
    )
)

In [ ]:
# ================================================================
# 48.14 — RECENT 30-DAY MOMENTUM
# ================================================================

fcv_feature_table[
    "recent30_revenue_share"
] = (
    fcv_feature_table[
        "r30_revenue"
    ]
    /
    fcv_feature_table[
        "h180_revenue"
    ]
)


fcv_feature_table[
    "recent30_order_share"
] = (
    fcv_feature_table[
        "r30_orders"
    ]
    /
    fcv_feature_table[
        "h180_orders"
    ]
)

In [ ]:
# ================================================================
# 48.15 — FREQUENCY
# ================================================================

fcv_feature_table[
    "h180_orders_per_30_days"
] = (
    fcv_feature_table[
        "h180_orders"
    ]
    /
    6
)


fcv_feature_table[
    "h180_active_day_rate"
] = (
    fcv_feature_table[
        "h180_active_days"
    ]
    /
    180
)


fcv_feature_table[
    "r90_active_day_rate"
] = (
    fcv_feature_table[
        "r90_active_days"
    ]
    /
    90
)


fcv_feature_table[
    "p90_active_day_rate"
] = (
    fcv_feature_table[
        "p90_active_days"
    ]
    /
    90
)

In [ ]:
# ================================================================
# 48.16 — PRODUCT BREADTH
# ================================================================

fcv_feature_table[
    "product_breadth_change"
] = (
    fcv_feature_table[
        "r90_unique_products"
    ]
    -
    fcv_feature_table[
        "p90_unique_products"
    ]
)

In [ ]:
# ================================================================
# 48.17 — LOG TRANSFORMS
# ================================================================

fcv_feature_table[
    "log1p_h180_revenue"
] = np.log1p(
    fcv_feature_table[
        "h180_revenue"
    ]
)


fcv_feature_table[
    "log1p_r90_revenue"
] = np.log1p(
    fcv_feature_table[
        "r90_revenue"
    ]
)


fcv_feature_table[
    "log1p_p90_revenue"
] = np.log1p(
    fcv_feature_table[
        "p90_revenue"
    ]
)


fcv_feature_table[
    "log1p_h180_orders"
] = np.log1p(
    fcv_feature_table[
        "h180_orders"
    ]
)

In [ ]:
# ================================================================
# 48.18 — CALENDAR
# ================================================================

fcv_feature_table[
    "prediction_month"
] = (
    fcv_feature_table[
        "prediction_date"
    ]
    .dt.month
)


fcv_feature_table[
    "prediction_month_sin"
] = np.sin(
    2
    *
    np.pi
    *
    fcv_feature_table[
        "prediction_month"
    ]
    /
    12
)


fcv_feature_table[
    "prediction_month_cos"
] = np.cos(
    2
    *
    np.pi
    *
    fcv_feature_table[
        "prediction_month"
    ]
    /
    12
)

In [ ]:
# ================================================================
# 48.19 — FCV FEATURE REGISTER
# ================================================================

fcv_numeric_features = [
    # 180-day scale
    "h180_revenue",
    "h180_orders",
    "h180_active_days",
    "h180_units",
    "h180_unique_products",
    "h180_average_unit_price",
    "h180_maximum_unit_price",
    "h180_average_order_value",
    "h180_revenue_per_active_day",
    "h180_units_per_order",
    "h180_products_per_order",

    # Recent 90 days
    "r90_revenue",
    "r90_orders",
    "r90_active_days",
    "r90_units",
    "r90_unique_products",
    "r90_average_order_value",
    "r90_active_day_rate",

    # Prior 90 days
    "p90_revenue",
    "p90_orders",
    "p90_active_days",
    "p90_units",
    "p90_unique_products",
    "p90_average_order_value",
    "p90_active_day_rate",

    # Recent 30 days
    "r30_revenue",
    "r30_orders",
    "r30_active_days",
    "r30_units",

    # Recency/activity
    "days_since_last_purchase",
    "active_in_last_30d",
    "active_in_last_90d",

    # Momentum
    "r90_minus_p90_revenue",
    "r90_to_p90_revenue_ratio",
    "log_revenue_momentum",
    "recent30_revenue_share",
    "recent30_order_share",
    "product_breadth_change",

    # Cadence
    "h180_average_gap_days",
    "h180_maximum_gap_days",
    "h180_minimum_gap_days",
    "h180_orders_per_30_days",
    "h180_active_day_rate",

    # Transforms
    "log1p_h180_revenue",
    "log1p_r90_revenue",
    "log1p_p90_revenue",
    "log1p_h180_orders",

    # Calendar
    "prediction_month_sin",
    "prediction_month_cos",
]


fcv_categorical_features = [
    "last_region",
]


fcv_model_features = (
    fcv_numeric_features
    +
    fcv_categorical_features
)


print(
    f"FCV model features: "
    f"{len(fcv_model_features)}"
)

In [ ]:
# ================================================================
# 48.20 — FUTURE INFORMATION AUDIT
# ================================================================

forbidden_fcv_features = [
    "future_90d_revenue",
    "future_90d_orders",
    "future_90d_active_days",
    "future_90d_units",
    "future_active",
    "fcv90",
    "outcome_end",
]


fcv_feature_overlap = (
    set(
        fcv_model_features
    )
    &
    set(
        forbidden_fcv_features
    )
)


assert not fcv_feature_overlap, (
    f"Future leakage detected: "
    f"{fcv_feature_overlap}"
)


print(
    "✓ No future-window variables appear in the feature register."
)

In [ ]:
# ================================================================
# 48.21 — FEATURE MISSINGNESS
# ================================================================

fcv_feature_missingness = (
    fcv_feature_table[
        fcv_model_features
    ]
    .isna()
    .mean()
    .mul(100)
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    fcv_feature_missingness
)

In [ ]:
# ================================================================
# 48.22 — FCV MODELLING TABLE
# ================================================================

fcv_model_dataset = (
    fcv_feature_table[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "outcome_end",
            "future_active",
            "fcv90",

            *fcv_model_features,
        ]
    ]
    .copy()
    .sort_values(
        [
            "prediction_date",
            "customer",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"FCV modelling episodes: "
    f"{len(fcv_model_dataset):,}"
)

### 48.23 — Purged temporal split
We'll maintain the discipline from Notebook 4.

**Train**
>May–Nov 2010

**Embargo**
>Dec 2010–Feb 2011

**Validation**
>Mar–Apr 2011

**Embargo**
>May–Jun 2011

**Test**
>Jul–Aug 2011

In [ ]:
# ================================================================
# 48.23 — TEMPORAL SPLIT DEFINITIONS
# ================================================================

FCV_TRAIN_START = pd.Period(
    "2010-05",
    freq="M"
)

FCV_TRAIN_END = pd.Period(
    "2010-11",
    freq="M"
)


FCV_VALIDATION_START = pd.Period(
    "2011-03",
    freq="M"
)

FCV_VALIDATION_END = pd.Period(
    "2011-04",
    freq="M"
)


FCV_TEST_START = pd.Period(
    "2011-07",
    freq="M"
)

FCV_TEST_END = pd.Period(
    "2011-08",
    freq="M"
)

In [ ]:
# ================================================================
# 48.24 — CREATE TRAIN / VALIDATION / TEST
# ================================================================

fcv_train_data = (
    fcv_model_dataset.loc[
        fcv_model_dataset[
            "prediction_period"
        ]
        .between(
            FCV_TRAIN_START,
            FCV_TRAIN_END
        )
    ]
    .copy()
)


fcv_validation_data = (
    fcv_model_dataset.loc[
        fcv_model_dataset[
            "prediction_period"
        ]
        .between(
            FCV_VALIDATION_START,
            FCV_VALIDATION_END
        )
    ]
    .copy()
)


fcv_test_data = (
    fcv_model_dataset.loc[
        fcv_model_dataset[
            "prediction_period"
        ]
        .between(
            FCV_TEST_START,
            FCV_TEST_END
        )
    ]
    .copy()
)

In [ ]:
# ================================================================
# 48.25 — EMBARGO POPULATIONS
# ================================================================

fcv_embargo_1 = (
    fcv_model_dataset.loc[
        fcv_model_dataset[
            "prediction_period"
        ]
        .between(
            pd.Period(
                "2010-12",
                freq="M"
            ),
            pd.Period(
                "2011-02",
                freq="M"
            )
        )
    ]
)


fcv_embargo_2 = (
    fcv_model_dataset.loc[
        fcv_model_dataset[
            "prediction_period"
        ]
        .between(
            pd.Period(
                "2011-05",
                freq="M"
            ),
            pd.Period(
                "2011-06",
                freq="M"
            )
        )
    ]
)


print(
    f"Embargo 1 episodes: "
    f"{len(fcv_embargo_1):,}"
)

print(
    f"Embargo 2 episodes: "
    f"{len(fcv_embargo_2):,}"
)

In [ ]:
# ================================================================
# 48.26 — DEVELOPMENT SPLIT SUMMARY
# ================================================================

fcv_development_split_summary = pd.DataFrame(
    [
        {
            "Dataset":
                "Train",

            "Episodes":
                len(
                    fcv_train_data
                ),

            "Unique Customers":
                fcv_train_data[
                    "customer"
                ].nunique(),

            "Start":
                fcv_train_data[
                    "prediction_period"
                ].min(),

            "End":
                fcv_train_data[
                    "prediction_period"
                ].max(),

            "Future Activity Rate":
                fcv_train_data[
                    "future_active"
                ].mean(),

            "Mean FCV90":
                fcv_train_data[
                    "fcv90"
                ].mean(),

            "Median FCV90":
                fcv_train_data[
                    "fcv90"
                ].median(),
        },

        {
            "Dataset":
                "Validation",

            "Episodes":
                len(
                    fcv_validation_data
                ),

            "Unique Customers":
                fcv_validation_data[
                    "customer"
                ].nunique(),

            "Start":
                fcv_validation_data[
                    "prediction_period"
                ].min(),

            "End":
                fcv_validation_data[
                    "prediction_period"
                ].max(),

            "Future Activity Rate":
                fcv_validation_data[
                    "future_active"
                ].mean(),

            "Mean FCV90":
                fcv_validation_data[
                    "fcv90"
                ].mean(),

            "Median FCV90":
                fcv_validation_data[
                    "fcv90"
                ].median(),
        },
    ]
)


fcv_development_split_summary[
    "Future Activity Rate (%)"
] = (
    fcv_development_split_summary[
        "Future Activity Rate"
    ]
    *
    100
)


display(
    fcv_development_split_summary
)

In [ ]:
# ================================================================
# 48.27 — TEST GOVERNANCE
# ================================================================

FCV_TEST_SET_OPENED = False


print(
    f"Test episodes reserved: "
    f"{len(fcv_test_data):,}"
)

print(
    f"Test prediction period: "
    f"{fcv_test_data['prediction_period'].min()} "
    f"to "
    f"{fcv_test_data['prediction_period'].max()}"
)

print(
    "Test future activity and FCV90 outcomes remain intentionally uninspected."
)

In [ ]:
# ================================================================
# 48.28 — STAGE A MATRICES
# ================================================================

X_fcv_train = (
    fcv_train_data[
        fcv_model_features
    ]
    .copy()
)


y_fcv_activity_train = (
    fcv_train_data[
        "future_active"
    ]
    .copy()
)


X_fcv_validation = (
    fcv_validation_data[
        fcv_model_features
    ]
    .copy()
)


y_fcv_activity_validation = (
    fcv_validation_data[
        "future_active"
    ]
    .copy()
)


X_fcv_test = (
    fcv_test_data[
        fcv_model_features
    ]
    .copy()
)


y_fcv_activity_test = (
    fcv_test_data[
        "future_active"
    ]
    .copy()
)

In [ ]:
# ================================================================
# 48.29 — STAGE B POSITIVE TARGET POPULATIONS
# ================================================================

fcv_positive_train_data = (
    fcv_train_data.loc[
        fcv_train_data[
            "fcv90"
        ]
        > 0
    ]
    .copy()
)


fcv_positive_validation_data = (
    fcv_validation_data.loc[
        fcv_validation_data[
            "fcv90"
        ]
        > 0
    ]
    .copy()
)


print(
    f"Positive Train episodes: "
    f"{len(fcv_positive_train_data):,}"
)

print(
    f"Positive Validation episodes: "
    f"{len(fcv_positive_validation_data):,}"
)

In [ ]:
# ================================================================
# 48.30 — POSITIVE VALUE REGRESSION TARGET
# ================================================================

X_fcv_positive_train = (
    fcv_positive_train_data[
        fcv_model_features
    ]
    .copy()
)


y_fcv_positive_train = (
    np.log1p(
        fcv_positive_train_data[
            "fcv90"
        ]
    )
)


X_fcv_positive_validation = (
    fcv_positive_validation_data[
        fcv_model_features
    ]
    .copy()
)


y_fcv_positive_validation = (
    np.log1p(
        fcv_positive_validation_data[
            "fcv90"
        ]
    )
)


print(
    "✓ Positive FCV90 target transformed using log1p."
)

In [ ]:
# ================================================================
# 48.31 — TRAIN → VALIDATION FEATURE DRIFT
# ================================================================

fcv_drift_records = []


for feature in fcv_numeric_features:

    train_values = pd.to_numeric(
        X_fcv_train[
            feature
        ],
        errors="coerce"
    )


    validation_values = pd.to_numeric(
        X_fcv_validation[
            feature
        ],
        errors="coerce"
    )


    train_mean = (
        train_values.mean()
    )


    validation_mean = (
        validation_values.mean()
    )


    pooled_std = np.sqrt(
        (
            train_values.std() ** 2
            +
            validation_values.std() ** 2
        )
        /
        2
    )


    if (
        pd.isna(pooled_std)
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            validation_mean
            -
            train_mean
        ) / pooled_std


    fcv_drift_records.append(
        {
            "feature":
                feature,

            "train_mean":
                train_mean,

            "validation_mean":
                validation_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(smd)
                if not pd.isna(smd)
                else np.nan,
        }
    )


fcv_feature_drift = (
    pd.DataFrame(
        fcv_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    fcv_feature_drift
)

In [ ]:
# ================================================================
# 48.32 — HURDLE TARGET SUMMARY
# ================================================================

hurdle_target_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
        ],

        "All Episodes": [
            len(
                fcv_train_data
            ),

            len(
                fcv_validation_data
            ),
        ],

        "Future Active Episodes": [
            fcv_train_data[
                "future_active"
            ].sum(),

            fcv_validation_data[
                "future_active"
            ].sum(),
        ],

        "Future Activity Rate": [
            fcv_train_data[
                "future_active"
            ].mean(),

            fcv_validation_data[
                "future_active"
            ].mean(),
        ],

        "Positive FCV90 Median": [
            fcv_positive_train_data[
                "fcv90"
            ].median(),

            fcv_positive_validation_data[
                "fcv90"
            ].median(),
        ],

        "Positive FCV90 Mean": [
            fcv_positive_train_data[
                "fcv90"
            ].mean(),

            fcv_positive_validation_data[
                "fcv90"
            ].mean(),
        ],
    }
)


display(
    hurdle_target_summary
)

In [ ]:
# ================================================================
# 48.33 — STEP 48 COMPLETION CHECK
# ================================================================

assert (
    len(
        fcv_train_data
    )
    > 0
)


assert (
    len(
        fcv_validation_data
    )
    > 0
)


assert (
    len(
        fcv_test_data
    )
    > 0
)


assert (
    FCV_TEST_SET_OPENED
    is False
)


assert (
    fcv_train_data[
        "fcv90"
    ]
    .ge(0)
    .all()
)


assert (
    fcv_validation_data[
        "fcv90"
    ]
    .ge(0)
    .all()
)


assert (
    set(
        fcv_train_data[
            "prediction_period"
        ]
    )
    .isdisjoint(
        set(
            fcv_validation_data[
                "prediction_period"
            ]
        )
    )
)


assert (
    set(
        fcv_validation_data[
            "prediction_period"
        ]
    )
    .isdisjoint(
        set(
            fcv_test_data[
                "prediction_period"
            ]
        )
    )
)


print(
    "✓ Step 48 complete."
)

print(
    "✓ Leakage-safe FCV features constructed."
)

print(
    "✓ Stage A future-activity target prepared."
)

print(
    "✓ Stage B positive-value regression target prepared."
)

print(
    "✓ Purged chronological populations created."
)

print(
    "✓ Final Test outcomes remain unopened."
)

### Step 48 Summary — FCV Feature Engineering and Temporal Architecture

Step 48 constructed a leakage-safe feature architecture for the two-part Future Customer Value model.

### Historical Reconstruction Was Exact

The reconstructed 180-day customer revenue matched the original eligibility-period revenue exactly.

The maximum absolute difference was:

> **0.00000000**

This confirms that feature construction and prediction-population construction are fully aligned.

### Missingness Is Primarily Structural

Approximately:

- **48.1%** of observations lack 180-day purchase-gap statistics;
- **31.4%** lack previous-90-day average order value and direct current/prior revenue-ratio measures;
- **30.4%** lack recent-90-day average order value.

These observations should remain in the modelling population.

The missingness mainly reflects customers without sufficient transactions in the relevant windows rather than data-quality failure.

Missingness will therefore be handled through preprocessing and missingness indicators.

### Development Environment Changes Through Time

The Train population contains **20,584 prediction episodes** with approximately **51.7% future activity**.

The Validation population contains **6,346 episodes** with approximately **45.8% future activity**.

Mean FCV90 declines from approximately **618** during Train to **473** during Validation.

Median FCV90 declines from approximately **102** to **zero**.

The chronological Validation period therefore represents a weaker future-value environment than the original Train period.

### Positive Customer Value Also Changes

Among episodes generating positive future revenue:

- Train median positive FCV90 was approximately **524**;
- Validation median was approximately **452**;
- Train mean positive FCV90 was approximately **1,194**;
- Validation mean was approximately **1,034**.

Both hurdle components therefore require genuine temporal generalisation.

### Long-Term Customer Scale Is Relatively Stable

Several 180-day relationship variables show very little Train-to-Validation drift.

Examples include:

- historical 180-day revenue;
- active purchase days;
- order frequency;
- product breadth;
- average order value;
- and units.

These features appear to represent comparatively stable structural customer value.

### Recent Customer Behaviour Changes More Substantially

Recent activity variables show greater temporal movement.

Examples include:

- recency;
- recent revenue momentum;
- recent product-breadth change;
- recent-90-day revenue;
- and recent activity indicators.

This suggests that future customer value depends on both:

> **structural customer scale**

and:

> **current behavioural momentum**

### Calendar Drift Is Extremely Large

Cyclical month encoding shows very large Train-to-Validation drift because the periods occupy different parts of the annual cycle.

Calendar information may contain legitimate seasonal signal, but models using calendar features should be compared with no-calendar challengers before final selection.

### Hurdle Architecture Remains Appropriate

The final architecture remains:

1. **Stage A — Future Activity Classification**
2. **Stage B — Positive FCV90 Regression**
3. **Combined Expected FCV90**

The final Test population of **5,534 episodes** remains unopened.



## Step 49 — Stage A: Future Activity Classification

Stage A estimates:

> **P(FCV90 > 0)**

or the probability that an eligible customer generates any qualifying merchandise revenue during the following 90 days.

This is the first component of the Future Customer Value hurdle model.

The classification model must distinguish customers likely to remain economically active from customers likely to generate zero future revenue.

### Primary Evaluation Metric

PR-AUC is the primary discrimination metric.

Because future activity prevalence is substantial, the analysis will also evaluate:

- ROC-AUC;
- Brier score;
- Precision;
- Recall;
- F1;
- Lift@10%;
- Lift@20%;
- Capture@10%;
- Capture@20%.

### Feature-Family Diagnostics

Because calendar variables show substantial temporal drift, a no-calendar challenger will be evaluated explicitly.

A structural-value benchmark will also test how much predictive power is available from relatively stable 180-day customer-scale variables alone.

The final Test population remains unopened.

In [ ]:
# ================================================================
# 49.2 — STAGE A IMPORTS
# ================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
)

import numpy as np
import pandas as pd

In [ ]:
# ================================================================
# 49.3 — TEST GOVERNANCE
# ================================================================

assert FCV_TEST_SET_OPENED is False


print(
    "✓ FCV Test population remains unopened."
)

print(
    "✓ Stage A uses Train + Validation only."
)

In [ ]:
# ================================================================
# 49.4 — STAGE A PREVALENCE
# ================================================================

fcv_activity_prevalence_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
        ],

        "Episodes": [
            len(y_fcv_activity_train),
            len(y_fcv_activity_validation),
        ],

        "Future Activity Rate": [
            y_fcv_activity_train.mean(),
            y_fcv_activity_validation.mean(),
        ],
    }
)


fcv_activity_prevalence_summary[
    "Future Activity Rate (%)"
] = (
    fcv_activity_prevalence_summary[
        "Future Activity Rate"
    ]
    *
    100
)


display(
    fcv_activity_prevalence_summary
)

In [ ]:
# ================================================================
# 49.5 — LIFT FUNCTION
# ================================================================

def fcv_lift_at_fraction(
    y_true,
    probabilities,
    fraction=0.10
):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)


    if np.allclose(
        probabilities,
        probabilities[0]
    ):
        return 1.0


    evaluation = pd.DataFrame(
        {
            "actual": y_true,
            "probability": probabilities,
        }
    ).sort_values(
        "probability",
        ascending=False,
        kind="mergesort"
    ).reset_index(drop=True)


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    overall_rate = (
        evaluation["actual"].mean()
    )


    if overall_rate == 0:
        return np.nan


    top_rate = (
        evaluation
        .head(n_top)["actual"]
        .mean()
    )


    return (
        top_rate
        /
        overall_rate
    )

In [ ]:
# ================================================================
# 49.6 — CAPTURE FUNCTION
# ================================================================

def fcv_capture_at_fraction(
    y_true,
    probabilities,
    fraction=0.10
):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)


    if np.allclose(
        probabilities,
        probabilities[0]
    ):
        return fraction


    evaluation = pd.DataFrame(
        {
            "actual": y_true,
            "probability": probabilities,
        }
    ).sort_values(
        "probability",
        ascending=False,
        kind="mergesort"
    ).reset_index(drop=True)


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    total_positives = (
        evaluation["actual"].sum()
    )


    if total_positives == 0:
        return np.nan


    return (
        evaluation
        .head(n_top)["actual"]
        .sum()
        /
        total_positives
    )

In [ ]:
# ================================================================
# 49.7 — CLASSIFICATION EVALUATION
# ================================================================

def evaluate_fcv_activity_model(
    model_name,
    y_true,
    probabilities,
    threshold=0.50
):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)


    predictions = (
        probabilities
        >=
        threshold
    ).astype(int)


    return pd.DataFrame(
        {
            "Model": [
                model_name
            ],

            "PR-AUC": [
                average_precision_score(
                    y_true,
                    probabilities
                )
            ],

            "ROC-AUC": [
                roc_auc_score(
                    y_true,
                    probabilities
                )
            ],

            "Brier Score": [
                brier_score_loss(
                    y_true,
                    probabilities
                )
            ],

            "Precision": [
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Recall": [
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "F1": [
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Lift @ 10%": [
                fcv_lift_at_fraction(
                    y_true,
                    probabilities,
                    0.10
                )
            ],

            "Lift @ 20%": [
                fcv_lift_at_fraction(
                    y_true,
                    probabilities,
                    0.20
                )
            ],

            "Capture @ 10%": [
                fcv_capture_at_fraction(
                    y_true,
                    probabilities,
                    0.10
                )
            ],

            "Capture @ 20%": [
                fcv_capture_at_fraction(
                    y_true,
                    probabilities,
                    0.20
                )
            ],
        }
    )

In [ ]:
# ================================================================
# 49.8 — PREVALENCE BASELINE
# ================================================================

fcv_train_activity_prevalence = (
    y_fcv_activity_train.mean()
)


fcv_validation_prevalence_probabilities = np.full(
    len(y_fcv_activity_validation),
    fcv_train_activity_prevalence
)


fcv_prevalence_baseline_results = (
    evaluate_fcv_activity_model(
        model_name="Training Prevalence Baseline",
        y_true=y_fcv_activity_validation,
        probabilities=fcv_validation_prevalence_probabilities,
    )
)


display(
    fcv_prevalence_baseline_results
)

In [ ]:
# ================================================================
# 49.9 — RECENCY-ONLY LOGISTIC
# ================================================================

fcv_recency_features = [
    "days_since_last_purchase"
]


fcv_recency_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            fcv_recency_features,
        )
    ]
)


fcv_recency_logistic = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_recency_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)


fcv_recency_logistic.fit(
    X_fcv_train[
        fcv_recency_features
    ],
    y_fcv_activity_train
)


validation_fcv_recency_probabilities = (
    fcv_recency_logistic
    .predict_proba(
        X_fcv_validation[
            fcv_recency_features
        ]
    )[:, 1]
)


fcv_recency_results = (
    evaluate_fcv_activity_model(
        model_name="Recency-Only Logistic",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_recency_probabilities,
    )
)


display(
    fcv_recency_results
)

### 49.10 — Structural-value benchmark
This intentionally uses relatively stable 180-day relationship measures.

In [ ]:
# ================================================================
# 49.10 — STRUCTURAL VALUE FEATURES
# ================================================================

fcv_structural_features = [
    "log1p_h180_revenue",
    "log1p_h180_orders",
    "h180_active_days",
    "h180_unique_products",
    "h180_average_order_value",
    "h180_revenue_per_active_day",
    "h180_average_gap_days",
    "days_since_last_purchase",
]

In [ ]:
fcv_structural_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            fcv_structural_features,
        )
    ]
)

In [ ]:
fcv_structural_logistic = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_structural_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)


fcv_structural_logistic.fit(
    X_fcv_train[
        fcv_structural_features
    ],
    y_fcv_activity_train
)


validation_fcv_structural_probabilities = (
    fcv_structural_logistic
    .predict_proba(
        X_fcv_validation[
            fcv_structural_features
        ]
    )[:, 1]
)


fcv_structural_results = (
    evaluate_fcv_activity_model(
        model_name="Structural-Value Logistic",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_structural_probabilities,
    )
)


display(
    fcv_structural_results
)

In [ ]:
# ================================================================
# 49.11 — FULL PREPROCESSOR
# ================================================================

fcv_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),
    ]
)


fcv_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        ),
    ]
)


fcv_full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            fcv_numeric_transformer,
            fcv_numeric_features
        ),

        (
            "categorical",
            fcv_categorical_transformer,
            fcv_categorical_features
        ),
    ]
)

### 49.12 — Full Logistic Regression

In [ ]:
# ================================================================
# 49.12 — FULL LOGISTIC REGRESSION
# ================================================================

fcv_full_logistic = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_full_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


fcv_full_logistic.fit(
    X_fcv_train,
    y_fcv_activity_train
)


validation_fcv_full_logistic_probabilities = (
    fcv_full_logistic
    .predict_proba(
        X_fcv_validation
    )[:, 1]
)


fcv_full_logistic_results = (
    evaluate_fcv_activity_model(
        model_name="Full Logistic Regression",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_full_logistic_probabilities,
    )
)


display(
    fcv_full_logistic_results
)

### 49.13 — No-calendar Logistic
Because calendar SMD is extreme.

In [ ]:
# ================================================================
# 49.13 — NO-CALENDAR LOGISTIC
# ================================================================

fcv_no_calendar_numeric_features = [
    feature
    for feature in fcv_numeric_features
    if feature not in [
        "prediction_month_sin",
        "prediction_month_cos",
    ]
]


fcv_no_calendar_features = (
    fcv_no_calendar_numeric_features
    +
    fcv_categorical_features
)

In [ ]:
fcv_no_calendar_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            fcv_no_calendar_numeric_features
        ),

        (
            "categorical",
            fcv_categorical_transformer,
            fcv_categorical_features
        ),
    ]
)

In [ ]:
fcv_no_calendar_logistic = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_no_calendar_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


fcv_no_calendar_logistic.fit(
    X_fcv_train[
        fcv_no_calendar_features
    ],
    y_fcv_activity_train
)


validation_fcv_no_calendar_probabilities = (
    fcv_no_calendar_logistic
    .predict_proba(
        X_fcv_validation[
            fcv_no_calendar_features
        ]
    )[:, 1]
)


fcv_no_calendar_results = (
    evaluate_fcv_activity_model(
        model_name="No-Calendar Logistic",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_no_calendar_probabilities,
    )
)


display(
    fcv_no_calendar_results
)

### 49.14 — Tree preprocessor
No scaling required.

In [ ]:
# ================================================================
# 49.14 — TREE PREPROCESSOR
# ================================================================

fcv_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    )
                ]
            ),
            fcv_numeric_features
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    ),
                ]
            ),
            fcv_categorical_features
        ),
    ]
)

### 49.15 — Random Forest

In [ ]:
# ================================================================
# 49.15 — RANDOM FOREST
# ================================================================

fcv_activity_random_forest = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_tree_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


fcv_activity_random_forest.fit(
    X_fcv_train,
    y_fcv_activity_train
)


validation_fcv_rf_probabilities = (
    fcv_activity_random_forest
    .predict_proba(
        X_fcv_validation
    )[:, 1]
)


fcv_rf_results = (
    evaluate_fcv_activity_model(
        model_name="Full Random Forest",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_rf_probabilities,
    )
)


display(
    fcv_rf_results
)

### 49.16 — Gradient Boosting

In [ ]:
# ================================================================
# 49.16 — GRADIENT BOOSTING
# ================================================================

fcv_activity_gradient_boosting = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_tree_preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=20,
                random_state=42
            )
        ),
    ]
)


fcv_activity_gradient_boosting.fit(
    X_fcv_train,
    y_fcv_activity_train
)


validation_fcv_gb_probabilities = (
    fcv_activity_gradient_boosting
    .predict_proba(
        X_fcv_validation
    )[:, 1]
)


fcv_gb_results = (
    evaluate_fcv_activity_model(
        model_name="Full Gradient Boosting",
        y_true=y_fcv_activity_validation,
        probabilities=validation_fcv_gb_probabilities,
    )
)


display(
    fcv_gb_results
)

In [ ]:
# ================================================================
# 49.17 — MODEL COMPARISON
# ================================================================

stage_a_model_comparison = (
    pd.concat(
        [
            fcv_prevalence_baseline_results,
            fcv_recency_results,
            fcv_structural_results,
            fcv_full_logistic_results,
            fcv_no_calendar_results,
            fcv_rf_results,
            fcv_gb_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 20%",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    stage_a_model_comparison
)

In [ ]:
# ================================================================
# 49.18 — PREVALENCE-ADJUSTED PR-AUC
# ================================================================

fcv_validation_activity_prevalence = (
    y_fcv_activity_validation.mean()
)


stage_a_model_comparison[
    "PR-AUC / Prevalence"
] = (
    stage_a_model_comparison[
        "PR-AUC"
    ]
    /
    fcv_validation_activity_prevalence
)


stage_a_model_comparison[
    "Normalised PR-AUC"
] = (
    (
        stage_a_model_comparison[
            "PR-AUC"
        ]
        -
        fcv_validation_activity_prevalence
    )
    /
    (
        1
        -
        fcv_validation_activity_prevalence
    )
)


display(
    stage_a_model_comparison
)

In [ ]:
# ================================================================
# 49.19 — PRACTICAL EQUIVALENCE BAND
# ================================================================

FCV_ACTIVITY_PR_AUC_TOLERANCE = 0.005


best_stage_a_pr_auc = (
    stage_a_model_comparison[
        "PR-AUC"
    ]
    .max()
)


stage_a_model_comparison[
    "Within_PR_AUC_Equivalence_Band"
] = (
    stage_a_model_comparison[
        "PR-AUC"
    ]
    >=
    (
        best_stage_a_pr_auc
        -
        FCV_ACTIVITY_PR_AUC_TOLERANCE
    )
)


display(
    stage_a_model_comparison
)

In [ ]:
# ================================================================
# 49.20 — DECILE PERFORMANCE
# ================================================================

def create_fcv_activity_deciles(
    y_true,
    probabilities,
    n_deciles=10
):

    evaluation = pd.DataFrame(
        {
            "actual": np.asarray(y_true),
            "probability": np.asarray(probabilities),
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(drop=True)
    )


    evaluation[
        "score_decile"
    ] = (
        pd.qcut(
            np.arange(
                len(evaluation)
            ),
            q=n_deciles,
            labels=False
        )
        + 1
    )


    overall_rate = (
        evaluation[
            "actual"
        ].mean()
    )


    result = (
        evaluation
        .groupby(
            "score_decile",
            as_index=False
        )
        .agg(
            episodes=(
                "actual",
                "size"
            ),

            actual_activity_rate=(
                "actual",
                "mean"
            ),

            average_predicted_probability=(
                "probability",
                "mean"
            ),

            active_episodes=(
                "actual",
                "sum"
            ),
        )
    )


    result[
        "actual_activity_rate_pct"
    ] = (
        result[
            "actual_activity_rate"
        ]
        *
        100
    )


    result[
        "average_predicted_probability_pct"
    ] = (
        result[
            "average_predicted_probability"
        ]
        *
        100
    )


    result[
        "lift"
    ] = (
        result[
            "actual_activity_rate"
        ]
        /
        overall_rate
    )


    return result

In [ ]:
# ================================================================
# 49.21 — DECILE TABLES
# ================================================================

fcv_full_logistic_deciles = (
    create_fcv_activity_deciles(
        y_fcv_activity_validation,
        validation_fcv_full_logistic_probabilities
    )
)


fcv_rf_deciles = (
    create_fcv_activity_deciles(
        y_fcv_activity_validation,
        validation_fcv_rf_probabilities
    )
)


fcv_gb_deciles = (
    create_fcv_activity_deciles(
        y_fcv_activity_validation,
        validation_fcv_gb_probabilities
    )
)


display(
    fcv_full_logistic_deciles
)

display(
    fcv_rf_deciles
)

display(
    fcv_gb_deciles
)

### 49.22 — Financial relevance of Stage A ranking
This is important for FCV.

A classification model could rank future activity correctly but still fail to identify where future revenue lies.

So let's measure the proportion of actual Validation FCV90 concentrated within the activity score ranking.

In [ ]:
# ================================================================
# 49.22 — FUTURE REVENUE CAPTURE BY ACTIVITY SCORE
# ================================================================

def future_revenue_capture_at_fraction(
    actual_revenue,
    score,
    fraction
):

    evaluation = pd.DataFrame(
        {
            "actual_revenue":
                np.asarray(
                    actual_revenue
                ),

            "score":
                np.asarray(
                    score
                ),
        }
    ).sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)


    n_top = int(
        np.ceil(
            len(evaluation)
            *
            fraction
        )
    )


    total_revenue = (
        evaluation[
            "actual_revenue"
        ]
        .sum()
    )


    if total_revenue == 0:
        return np.nan


    captured = (
        evaluation
        .head(n_top)[
            "actual_revenue"
        ]
        .sum()
    )


    return (
        captured
        /
        total_revenue
    )

In [ ]:
# ================================================================
# 49.23 — ACTIVITY SCORE ECONOMIC CAPTURE
# ================================================================

stage_a_economic_capture_records = []


stage_a_probability_sets = {
    "Full Logistic":
        validation_fcv_full_logistic_probabilities,

    "Random Forest":
        validation_fcv_rf_probabilities,

    "Gradient Boosting":
        validation_fcv_gb_probabilities,
}


for model_name, probabilities in (
    stage_a_probability_sets.items()
):

    for fraction in [
        0.10,
        0.20,
        0.30,
    ]:

        stage_a_economic_capture_records.append(
            {
                "Model":
                    model_name,

                "Customer Fraction":
                    fraction,

                "Future Revenue Capture":
                    future_revenue_capture_at_fraction(
                        actual_revenue=fcv_validation_data[
                            "fcv90"
                        ],
                        score=probabilities,
                        fraction=fraction
                    ),
            }
        )


stage_a_economic_capture = pd.DataFrame(
    stage_a_economic_capture_records
)


stage_a_economic_capture[
    "Future Revenue Capture (%)"
] = (
    stage_a_economic_capture[
        "Future Revenue Capture"
    ]
    *
    100
)


display(
    stage_a_economic_capture
)

In [ ]:
# ================================================================
# 49.24 — MODEL SCORE CORRELATIONS
# ================================================================

stage_a_score_correlation = (
    pd.DataFrame(
        {
            "Full_Logistic":
                validation_fcv_full_logistic_probabilities,

            "Random_Forest":
                validation_fcv_rf_probabilities,

            "Gradient_Boosting":
                validation_fcv_gb_probabilities,
        }
    )
    .corr(
        method="spearman"
    )
)


display(
    stage_a_score_correlation.round(3)
)

In [ ]:
# ================================================================
# 49.25 — STEP 49 COMPLETION CHECK
# ================================================================

assert FCV_TEST_SET_OPENED is False


for probabilities in [
    validation_fcv_full_logistic_probabilities,
    validation_fcv_rf_probabilities,
    validation_fcv_gb_probabilities,
]:

    assert (
        len(probabilities)
        ==
        len(
            y_fcv_activity_validation
        )
    )


print(
    "✓ Step 49 complete."
)

print(
    "✓ Future-activity prevalence baseline established."
)

print(
    "✓ Recency and structural-value benchmarks established."
)

print(
    "✓ Full Logistic evaluated."
)

print(
    "✓ No-calendar challenger evaluated."
)

print(
    "✓ Random Forest evaluated."
)

print(
    "✓ Gradient Boosting evaluated."
)

print(
    "✓ Stage A economic capture evaluated."
)

print(
    "✓ Final Test outcomes remain unopened."
)

### Step 49 Summary — Stage A Future Activity Models

Step 49 developed the first component of the Future Customer Value hurdle architecture.

Stage A estimates:

> **P(FCV90 > 0)**

or the probability that an eligible customer generates qualifying merchandise revenue during the following 90 days.

### Future Activity Is Strongly Predictable

Validation future-activity prevalence was approximately **45.8%**.

The strongest models achieved approximately:

- **Gradient Boosting PR-AUC: 0.753**
- **Random Forest PR-AUC: 0.751**
- **Logistic Regression PR-AUC: 0.750**

All three models fall within the predetermined 0.005 PR-AUC practical-equivalence range.

Future customer activity therefore contains substantial predictable structure.

### High-Probability Customers Are Strongly Concentrated

Gradient Boosting achieved approximately:

- **Lift@10%: 2.05×**
- **Lift@20%: 1.82×**

The highest-scored decile experienced approximately **93.9% future activity**, compared with overall activity of approximately **45.8%**.

The lowest-scored decile experienced only approximately **22.0% future activity**.

The model therefore creates strong operational separation between customers likely and unlikely to generate future revenue.

### Structural Customer Value Is Already Highly Informative

A comparatively simple model based primarily on stable 180-day relationship features achieved approximately:

- **PR-AUC: 0.739**
- **ROC-AUC: 0.743**
- **Lift@10%: 1.98×**

This confirms that longer-term customer economic structure contains substantial durable information about future purchasing activity.

### Recent Behaviour Adds Additional Predictive Signal

The full models improve upon the structural benchmark by incorporating:

- recent purchase activity;
- revenue momentum;
- recency;
- product breadth;
- and seasonal context.

This supports the emerging FCV architecture in which future economic value depends on both:

> **structural customer scale**

and:

> **current customer momentum**

### Calendar Is Helpful but Not Essential

Removing calendar features from Logistic Regression reduced PR-AUC only modestly, from approximately **0.750 to 0.746**.

The future-activity signal therefore does not depend exclusively on seasonal variables.

### Stage A Scores Already Concentrate Future Revenue

Although Stage A was trained only to predict future activity, its probability ranking also strongly concentrated actual future revenue.

Across the strongest models:

- the highest **10%** of customer scores captured approximately **52–53%** of Validation FCV90;
- the highest **20%** captured approximately **67%**;
- the highest **30%** captured approximately **75–76%**.

Future activity probability is therefore economically meaningful even before the positive-value regression component is introduced.

### Candidate Position

Three models remain serious Stage A contenders:

1. **Full Gradient Boosting**
2. **Full Random Forest**
3. **Full Logistic Regression**

Their validation discrimination is practically equivalent.

Final Stage A selection will therefore depend on:

- purged temporal robustness;
- probability calibration;
- commercial lift;
- future-revenue concentration;
- and model stability.

The final Test population remains unopened.


## Step 50 — Stage A Temporal Robustness and Champion Selection

Step 50 performs the final pre-Test evaluation of the three strongest Future Activity models:

1. Full Logistic Regression;
2. Full Random Forest;
3. Full Gradient Boosting.

Because these models are practically equivalent on Validation PR-AUC, selection should not depend on a very small single-period performance difference.

The models will therefore be compared using:

- purged historical validation;
- PR-AUC;
- ROC-AUC;
- Brier score;
- top-decile and top-20% lift;
- future-revenue capture;
- probability calibration;
- and paired bootstrap uncertainty.

Every historical training episode must have a fully matured 90-day target before the later validation month begins.

The final FCV Test population remains unopened.

In [ ]:
# ================================================================
# 50.2 — TEST GOVERNANCE
# ================================================================

assert FCV_TEST_SET_OPENED is False

print(
    "✓ FCV Test population remains unopened."
)

In [ ]:
# ================================================================
# 50.3 — TEMPORAL DIAGNOSTIC DATASET
# ================================================================

fcv_temporal_dataset = (
    fcv_model_dataset[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "outcome_end",
            "future_active",
            "fcv90",
            *fcv_model_features,
        ]
    ]
    .sort_values(
        [
            "prediction_date",
            "customer",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Temporal diagnostic episodes: "
    f"{len(fcv_temporal_dataset):,}"
)

### 50.4 — Historical validation months
Use:
- Sep 2010
- Oct 2010
- Nov 2010

In [ ]:
# ================================================================
# 50.4 — PURGED VALIDATION MONTHS
# ================================================================

FCV_PURGED_VALIDATION_PERIODS = [
    pd.Period(
        "2010-09",
        freq="M"
    ),
    pd.Period(
        "2010-10",
        freq="M"
    ),
    pd.Period(
        "2010-11",
        freq="M"
    ),
]


print(
    FCV_PURGED_VALIDATION_PERIODS
)

In [ ]:
# ================================================================
# 50.5 — CREATE PURGED TEMPORAL FOLDS
# ================================================================

fcv_purged_folds = []


for validation_period in (
    FCV_PURGED_VALIDATION_PERIODS
):

    validation_month_start = (
        validation_period
        .to_timestamp(
            how="start"
        )
    )


    validation_mask = (
        fcv_temporal_dataset[
            "prediction_period"
        ]
        ==
        validation_period
    )


    train_mask = (
        fcv_temporal_dataset[
            "outcome_end"
        ]
        <
        validation_month_start
    )


    fold_train = (
        fcv_temporal_dataset.loc[
            train_mask
        ]
        .copy()
    )


    fold_validation = (
        fcv_temporal_dataset.loc[
            validation_mask
        ]
        .copy()
    )


    if (
        len(fold_train) > 0
        and
        len(fold_validation) > 0
    ):

        fcv_purged_folds.append(
            {
                "validation_period":
                    validation_period,

                "train":
                    fold_train,

                "validation":
                    fold_validation,
            }
        )


print(
    f"Purged folds created: "
    f"{len(fcv_purged_folds)}"
)

In [ ]:
# ================================================================
# 50.6 — LABEL-AVAILABILITY AUDIT
# ================================================================

fcv_purged_fold_audit = []


for fold in fcv_purged_folds:

    fold_train = (
        fold[
            "train"
        ]
    )


    fold_validation = (
        fold[
            "validation"
        ]
    )


    fcv_purged_fold_audit.append(
        {
            "validation_period":
                str(
                    fold[
                        "validation_period"
                    ]
                ),

            "train_episodes":
                len(
                    fold_train
                ),

            "validation_episodes":
                len(
                    fold_validation
                ),

            "latest_training_prediction":
                fold_train[
                    "prediction_date"
                ]
                .max(),

            "latest_training_outcome_known":
                fold_train[
                    "outcome_end"
                ]
                .max(),

            "validation_prediction_date":
                fold_validation[
                    "prediction_date"
                ]
                .min(),

            "validation_activity_rate":
                fold_validation[
                    "future_active"
                ]
                .mean(),

            "validation_mean_fcv90":
                fold_validation[
                    "fcv90"
                ]
                .mean(),
        }
    )


fcv_purged_fold_audit = pd.DataFrame(
    fcv_purged_fold_audit
)


display(
    fcv_purged_fold_audit
)

In [ ]:
# ================================================================
# 50.7 — CLONE IMPORT
# ================================================================

from sklearn.base import clone

In [ ]:
# ================================================================
# 50.8 — STAGE A FINALISTS
# ================================================================

fcv_stage_a_candidates = {
    "Full Logistic Regression":
        fcv_full_logistic,

    "Full Random Forest":
        fcv_activity_random_forest,

    "Full Gradient Boosting":
        fcv_activity_gradient_boosting,
}

In [ ]:
# ================================================================
# 50.9 — PURGED TEMPORAL EVALUATION
# ================================================================

fcv_stage_a_temporal_records = []


for fold_number, fold in enumerate(
    fcv_purged_folds,
    start=1
):

    fold_train = fold[
        "train"
    ]

    fold_validation = fold[
        "validation"
    ]


    X_fold_train = (
        fold_train[
            fcv_model_features
        ]
    )


    y_fold_train = (
        fold_train[
            "future_active"
        ]
    )


    X_fold_validation = (
        fold_validation[
            fcv_model_features
        ]
    )


    y_fold_validation = (
        fold_validation[
            "future_active"
        ]
    )


    actual_future_revenue = (
        fold_validation[
            "fcv90"
        ]
    )


    for (
        model_name,
        base_model
    ) in (
        fcv_stage_a_candidates.items()
    ):

        fold_model = clone(
            base_model
        )


        fold_model.fit(
            X_fold_train,
            y_fold_train
        )


        probabilities = (
            fold_model
            .predict_proba(
                X_fold_validation
            )[:, 1]
        )


        fcv_stage_a_temporal_records.append(
            {
                "fold":
                    fold_number,

                "validation_period":
                    str(
                        fold[
                            "validation_period"
                        ]
                    ),

                "model":
                    model_name,

                "train_episodes":
                    len(
                        fold_train
                    ),

                "validation_episodes":
                    len(
                        fold_validation
                    ),

                "validation_prevalence":
                    y_fold_validation.mean(),

                "PR_AUC":
                    average_precision_score(
                        y_fold_validation,
                        probabilities
                    ),

                "ROC_AUC":
                    roc_auc_score(
                        y_fold_validation,
                        probabilities
                    ),

                "Brier":
                    brier_score_loss(
                        y_fold_validation,
                        probabilities
                    ),

                "Lift_10":
                    fcv_lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.10
                    ),

                "Lift_20":
                    fcv_lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.20
                    ),

                "Capture_20":
                    fcv_capture_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.20
                    ),

                "Revenue_Capture_10":
                    future_revenue_capture_at_fraction(
                        actual_revenue=
                            actual_future_revenue,

                        score=
                            probabilities,

                        fraction=
                            0.10
                    ),

                "Revenue_Capture_20":
                    future_revenue_capture_at_fraction(
                        actual_revenue=
                            actual_future_revenue,

                        score=
                            probabilities,

                        fraction=
                            0.20
                    ),
            }
        )


fcv_stage_a_temporal_results = pd.DataFrame(
    fcv_stage_a_temporal_records
)


display(
    fcv_stage_a_temporal_results
)

In [ ]:
# ================================================================
# 50.10 — TEMPORAL SUMMARY
# ================================================================

fcv_stage_a_temporal_summary = (
    fcv_stage_a_temporal_results
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        folds=(
            "fold",
            "nunique"
        ),

        mean_PR_AUC=(
            "PR_AUC",
            "mean"
        ),

        std_PR_AUC=(
            "PR_AUC",
            "std"
        ),

        min_PR_AUC=(
            "PR_AUC",
            "min"
        ),

        mean_ROC_AUC=(
            "ROC_AUC",
            "mean"
        ),

        mean_Brier=(
            "Brier",
            "mean"
        ),

        mean_Lift_10=(
            "Lift_10",
            "mean"
        ),

        min_Lift_10=(
            "Lift_10",
            "min"
        ),

        mean_Lift_20=(
            "Lift_20",
            "mean"
        ),

        min_Lift_20=(
            "Lift_20",
            "min"
        ),

        mean_Revenue_Capture_10=(
            "Revenue_Capture_10",
            "mean"
        ),

        mean_Revenue_Capture_20=(
            "Revenue_Capture_20",
            "mean"
        ),
    )
)


display(
    fcv_stage_a_temporal_summary
)

In [ ]:
# ================================================================
# 50.11 — HISTORICAL FOLD WINNERS
# ================================================================

fcv_stage_a_pr_winners = (
    fcv_stage_a_temporal_results.loc[
        fcv_stage_a_temporal_results
        .groupby(
            "validation_period"
        )[
            "PR_AUC"
        ]
        .idxmax(),
        [
            "validation_period",
            "model",
            "PR_AUC",
        ]
    ]
)


fcv_stage_a_revenue_winners = (
    fcv_stage_a_temporal_results.loc[
        fcv_stage_a_temporal_results
        .groupby(
            "validation_period"
        )[
            "Revenue_Capture_20"
        ]
        .idxmax(),
        [
            "validation_period",
            "model",
            "Revenue_Capture_20",
        ]
    ]
)


print(
    "PR-AUC winners:"
)

display(
    fcv_stage_a_pr_winners
)


print(
    "Revenue Capture @20% winners:"
)

display(
    fcv_stage_a_revenue_winners
)

In [ ]:
# ================================================================
# 50.12 — CALIBRATION IMPORT
# ================================================================

from sklearn.calibration import calibration_curve

In [ ]:
# ================================================================
# 50.13 — CALIBRATION ERROR
# ================================================================

def fcv_mean_absolute_calibration_error(
    y_true,
    probabilities,
    n_bins=10
):

    prob_true, prob_pred = (
        calibration_curve(
            y_true=y_true,
            y_prob=probabilities,
            n_bins=n_bins,
            strategy="quantile"
        )
    )


    return (
        np.mean(
            np.abs(
                prob_true
                -
                prob_pred
            )
        )
        *
        100
    )

In [ ]:
# ================================================================
# 50.14 — VALIDATION CALIBRATION SUMMARY
# ================================================================

fcv_stage_a_calibration_summary = pd.DataFrame(
    {
        "Model": [
            "Full Logistic Regression",
            "Full Random Forest",
            "Full Gradient Boosting",
        ],

        "Calibration Error (pp)": [
            fcv_mean_absolute_calibration_error(
                y_fcv_activity_validation,
                validation_fcv_full_logistic_probabilities
            ),

            fcv_mean_absolute_calibration_error(
                y_fcv_activity_validation,
                validation_fcv_rf_probabilities
            ),

            fcv_mean_absolute_calibration_error(
                y_fcv_activity_validation,
                validation_fcv_gb_probabilities
            ),
        ],
    }
)


display(
    fcv_stage_a_calibration_summary
)

In [ ]:
plt.figure(
    figsize=(9, 7)
)


for (
    model_name,
    probabilities
) in [
    (
        "Logistic",
        validation_fcv_full_logistic_probabilities
    ),

    (
        "Random Forest",
        validation_fcv_rf_probabilities
    ),

    (
        "Gradient Boosting",
        validation_fcv_gb_probabilities
    ),
]:

    observed, predicted = (
        calibration_curve(
            y_true=
                y_fcv_activity_validation,

            y_prob=
                probabilities,

            n_bins=10,

            strategy="quantile"
        )
    )


    plt.plot(
        predicted,
        observed,
        marker="o",
        label=model_name
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


plt.title(
    "Stage A Future Activity — Validation Calibration"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Future Activity"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 50.15 — PAIRED BOOTSTRAP
# ================================================================

def fcv_stage_a_paired_bootstrap(
    y_true,
    actual_revenue,
    probabilities_a,
    probabilities_b,
    n_bootstrap=2000,
    random_state=42
):

    y_true = np.asarray(
        y_true
    )

    actual_revenue = np.asarray(
        actual_revenue
    )

    probabilities_a = np.asarray(
        probabilities_a
    )

    probabilities_b = np.asarray(
        probabilities_b
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = y_true[
            indices
        ]


        if (
            np.unique(
                y_sample
            ).size
            < 2
        ):
            continue


        revenue_sample = (
            actual_revenue[
                indices
            ]
        )


        a = probabilities_a[
            indices
        ]


        b = probabilities_b[
            indices
        ]


        records.append(
            {
                "PR_AUC_Difference":
                    average_precision_score(
                        y_sample,
                        a
                    )
                    -
                    average_precision_score(
                        y_sample,
                        b
                    ),

                "ROC_AUC_Difference":
                    roc_auc_score(
                        y_sample,
                        a
                    )
                    -
                    roc_auc_score(
                        y_sample,
                        b
                    ),

                "Lift10_Difference":
                    fcv_lift_at_fraction(
                        y_sample,
                        a,
                        0.10
                    )
                    -
                    fcv_lift_at_fraction(
                        y_sample,
                        b,
                        0.10
                    ),

                "RevenueCapture20_Difference":
                    future_revenue_capture_at_fraction(
                        revenue_sample,
                        a,
                        0.20
                    )
                    -
                    future_revenue_capture_at_fraction(
                        revenue_sample,
                        b,
                        0.20
                    ),
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
# ================================================================
# 50.16 — GRADIENT BOOSTING VS RANDOM FOREST
# ================================================================

fcv_gb_vs_rf_bootstrap = (
    fcv_stage_a_paired_bootstrap(
        y_true=
            y_fcv_activity_validation,

        actual_revenue=
            fcv_validation_data[
                "fcv90"
            ],

        probabilities_a=
            validation_fcv_gb_probabilities,

        probabilities_b=
            validation_fcv_rf_probabilities,

        n_bootstrap=2000,

        random_state=42
    )
)

In [ ]:
# ================================================================
# 50.17 — GRADIENT BOOSTING VS LOGISTIC
# ================================================================

fcv_gb_vs_logistic_bootstrap = (
    fcv_stage_a_paired_bootstrap(
        y_true=
            y_fcv_activity_validation,

        actual_revenue=
            fcv_validation_data[
                "fcv90"
            ],

        probabilities_a=
            validation_fcv_gb_probabilities,

        probabilities_b=
            validation_fcv_full_logistic_probabilities,

        n_bootstrap=2000,

        random_state=42
    )
)

In [ ]:
# ================================================================
# 50.18 — BOOTSTRAP SUMMARY
# ================================================================

def summarise_fcv_stage_a_bootstrap(
    bootstrap,
    comparison_name
):

    output = []


    for metric in [
        "PR_AUC_Difference",
        "ROC_AUC_Difference",
        "Lift10_Difference",
        "RevenueCapture20_Difference",
    ]:

        output.append(
            {
                "Comparison":
                    comparison_name,

                "Metric":
                    metric,

                "Mean Difference":
                    bootstrap[
                        metric
                    ].mean(),

                "CI 2.5%":
                    bootstrap[
                        metric
                    ].quantile(
                        0.025
                    ),

                "CI 97.5%":
                    bootstrap[
                        metric
                    ].quantile(
                        0.975
                    ),

                "Probability First Model Better":
                    (
                        bootstrap[
                            metric
                        ]
                        > 0
                    ).mean(),
            }
        )


    return pd.DataFrame(
        output
    )

In [ ]:
fcv_stage_a_bootstrap_summary = pd.concat(
    [
        summarise_fcv_stage_a_bootstrap(
            fcv_gb_vs_rf_bootstrap,
            "Gradient Boosting - Random Forest"
        ),

        summarise_fcv_stage_a_bootstrap(
            fcv_gb_vs_logistic_bootstrap,
            "Gradient Boosting - Logistic"
        ),
    ],
    ignore_index=True
)


display(
    fcv_stage_a_bootstrap_summary
)

In [ ]:
# ================================================================
# 50.19 — FINAL STAGE A PRE-TEST SCORECARD
# ================================================================

fcv_stage_a_pretest_scorecard = pd.DataFrame(
    [
        {
            "Model":
                "Full Logistic Regression",

            "Validation PR-AUC":
                fcv_full_logistic_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                fcv_full_logistic_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                fcv_full_logistic_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                fcv_full_logistic_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                fcv_full_logistic_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Revenue Capture@20":
                future_revenue_capture_at_fraction(
                    fcv_validation_data[
                        "fcv90"
                    ],

                    validation_fcv_full_logistic_probabilities,

                    0.20
                ),

            "Calibration Error pp":
                fcv_stage_a_calibration_summary.loc[
                    fcv_stage_a_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full Logistic Regression",
                    "Calibration Error (pp)"
                ].iloc[0],
        },

        {
            "Model":
                "Full Random Forest",

            "Validation PR-AUC":
                fcv_rf_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                fcv_rf_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                fcv_rf_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                fcv_rf_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                fcv_rf_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Revenue Capture@20":
                future_revenue_capture_at_fraction(
                    fcv_validation_data[
                        "fcv90"
                    ],

                    validation_fcv_rf_probabilities,

                    0.20
                ),

            "Calibration Error pp":
                fcv_stage_a_calibration_summary.loc[
                    fcv_stage_a_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full Random Forest",
                    "Calibration Error (pp)"
                ].iloc[0],
        },

        {
            "Model":
                "Full Gradient Boosting",

            "Validation PR-AUC":
                fcv_gb_results[
                    "PR-AUC"
                ].iloc[0],

            "Validation ROC-AUC":
                fcv_gb_results[
                    "ROC-AUC"
                ].iloc[0],

            "Validation Brier":
                fcv_gb_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                fcv_gb_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                fcv_gb_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Revenue Capture@20":
                future_revenue_capture_at_fraction(
                    fcv_validation_data[
                        "fcv90"
                    ],

                    validation_fcv_gb_probabilities,

                    0.20
                ),

            "Calibration Error pp":
                fcv_stage_a_calibration_summary.loc[
                    fcv_stage_a_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Full Gradient Boosting",
                    "Calibration Error (pp)"
                ].iloc[0],
        },
    ]
)


display(
    fcv_stage_a_pretest_scorecard
)

In [ ]:
# ================================================================
# 50.20 — DECISION PENDING
# ================================================================

FCV_STAGE_A_MODEL_FROZEN = False

FINAL_FCV_STAGE_A_MODEL_NAME = None
FINAL_FCV_STAGE_A_MODEL = None


print(
    "Stage A model remains unfrozen "
    "pending Step 50 evidence review."
)

In [ ]:
# ================================================================
# 50.21 — STEP 50 COMPLETE
# ================================================================

assert FCV_TEST_SET_OPENED is False

assert (
    FCV_STAGE_A_MODEL_FROZEN
    is False
)


print(
    "✓ Step 50 diagnostics complete."
)

print(
    "✓ Purged historical validation completed."
)

print(
    "✓ Calibration evaluated."
)

print(
    "✓ Economic concentration evaluated."
)

print(
    "✓ Paired bootstrap completed."
)

print(
    "✓ Final FCV Test population remains unopened."
)

### Step 50 Summary — Stage A Temporal Robustness and Champion Selection

Step 50 performed purged historical validation, calibration analysis and paired bootstrap comparison for the three strongest Future Activity models.

### Historical Validation Was Label-Matured

Every training episode used in each historical fold had a fully completed 90-day target window before the later validation prediction date.

The analysis therefore avoids using future target information that would not have been available at the historical scoring point.

### Three Models Remained Strong

Full Logistic Regression, Random Forest and Gradient Boosting all demonstrated strong future-activity discrimination.

On the designated Validation population, PR-AUC was approximately:

- **Logistic Regression: 0.750**
- **Random Forest: 0.751**
- **Gradient Boosting: 0.753**

The models are therefore practically equivalent on the primary discrimination metric.

### Gradient Boosting Was Marginally Strongest on the Designated Validation Period

Gradient Boosting achieved the highest designated Validation:

- PR-AUC;
- ROC-AUC;
- Lift@10%;
- and future-revenue capture at 20%.

However, paired bootstrap confidence intervals for its differences versus Random Forest generally included zero.

The single-period advantage is therefore small rather than decisive.

### Random Forest Demonstrated Slightly Stronger Historical Robustness

Across the three purged historical folds, Random Forest produced approximately:

- **mean PR-AUC: 0.737**
- **mean Lift@20%: 1.73×**
- **mean future-revenue capture@20%: 63.7%**

It achieved the highest historical PR-AUC in two of the three folds and the highest Revenue Capture@20% in two of the three folds.

Its average historical Brier score was also slightly better than Gradient Boosting.

### Logistic Regression Was Less Suitable for the Hurdle Architecture

Although Logistic Regression ranked customers well, its Validation calibration error was approximately **14.9 percentage points**.

By comparison:

- Random Forest: approximately **4.3 pp**
- Gradient Boosting: approximately **3.9 pp**

Because Stage A probabilities will ultimately be multiplied by Stage B revenue estimates, credible probability calibration is particularly important.

Logistic Regression was therefore not selected as the final Stage A model.

## Stage A Champion

The frozen Stage A model is:

> **Full Random Forest**

Random Forest is selected because it provides the strongest overall balance of:

- high discrimination;
- good probability calibration;
- strong top-customer concentration;
- historical temporal stability;
- and future-revenue capture.

Gradient Boosting remains a strong challenger but does not demonstrate sufficient historical superiority to displace Random Forest.

The final FCV Test population remains unopened.

In [ ]:
# ================================================================
# 50.22 — FREEZE STAGE A CHAMPION
# ================================================================

FINAL_FCV_STAGE_A_MODEL_NAME = (
    "Full Random Forest"
)


FINAL_FCV_STAGE_A_MODEL = (
    fcv_activity_random_forest
)


FINAL_FCV_STAGE_A_FEATURES = (
    fcv_model_features.copy()
)


FCV_STAGE_A_MODEL_FROZEN = True


print(
    f"Frozen Stage A model: "
    f"{FINAL_FCV_STAGE_A_MODEL_NAME}"
)

print(
    f"Frozen feature count: "
    f"{len(FINAL_FCV_STAGE_A_FEATURES)}"
)

print(
    "✓ Stage A Future Activity specification frozen."
)

In [ ]:
# ================================================================
# 50.23 — STAGE A MODEL SPECIFICATION
# ================================================================

fcv_stage_a_frozen_specification = pd.DataFrame(
    {
        "Component": [
            "Prediction target",
            "Model",
            "Trees",
            "Maximum depth",
            "Minimum samples per leaf",
            "Maximum features",
            "Numeric missingness",
            "Categorical processing",
            "Primary metric",
            "Economic metric",
            "Validation design",
            "Test status",
        ],

        "Frozen Specification": [
            "Probability FCV90 > 0",
            "Random Forest Classifier",
            500,
            10,
            20,
            "sqrt",
            "Median imputation + missingness indicators",
            "Most-frequent imputation + one-hot encoding",
            "PR-AUC",
            "Future revenue captured by ranked customers",
            "Purged chronological validation",
            "UNOPENED",
        ],
    }
)


display(
    fcv_stage_a_frozen_specification
)

In [ ]:
# ================================================================
# 50.24 — STAGE A GOVERNANCE LOCK
# ================================================================

assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_TEST_SET_OPENED is False

assert (
    FINAL_FCV_STAGE_A_MODEL_NAME
    ==
    "Full Random Forest"
)


print(
    "✓ Stage A algorithm frozen."
)

print(
    "✓ Stage A hyperparameters frozen."
)

print(
    "✓ Stage A feature architecture frozen."
)

print(
    "✓ FCV Test outcomes remain unopened."
)

## Step 51 — Stage B: Positive Future Value Regression

Stage B estimates future revenue magnitude conditional on the customer generating positive revenue during the following 90 days.

The modelling target is:

> **FCV90 | FCV90 > 0**

Because positive FCV90 is strongly right-skewed, the primary statistical target is:

> **log(1 + FCV90)**

Models will be evaluated in both transformed and original revenue space.

### Statistical Evaluation

The analysis will consider:

- MAE;
- RMSE;
- log-space MAE;
- log-space RMSE;
- R²;
- and Spearman rank correlation.

### Economic Evaluation

Because future customer value is highly concentrated, model quality will also be evaluated using:

- actual future revenue captured by the highest predicted-value 10%;
- capture by the highest 20%;
- capture by the highest 30%;
- and calibration of predicted aggregate revenue.

The final Test population remains unopened.

In [ ]:
# ================================================================
# 51.2 — REGRESSION IMPORTS
# ================================================================

from sklearn.linear_model import (
    Ridge,
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from scipy.stats import spearmanr

In [ ]:
# ================================================================
# 51.3 — TEST GOVERNANCE
# ================================================================

assert FCV_TEST_SET_OPENED is False
assert FCV_STAGE_A_MODEL_FROZEN is True


print(
    "✓ Stage A remains frozen."
)

print(
    "✓ Final FCV Test remains unopened."
)

In [ ]:
# ================================================================
# 51.4 — POSITIVE FCV DEVELOPMENT POPULATIONS
# ================================================================

positive_fcv_development_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
        ],

        "Episodes": [
            len(
                fcv_positive_train_data
            ),

            len(
                fcv_positive_validation_data
            ),
        ],

        "Mean FCV90": [
            fcv_positive_train_data[
                "fcv90"
            ].mean(),

            fcv_positive_validation_data[
                "fcv90"
            ].mean(),
        ],

        "Median FCV90": [
            fcv_positive_train_data[
                "fcv90"
            ].median(),

            fcv_positive_validation_data[
                "fcv90"
            ].median(),
        ],
    }
)


display(
    positive_fcv_development_summary
)

In [ ]:
# ================================================================
# 51.5 — LINEAR REGRESSION PREPROCESSOR
# ================================================================

fcv_regression_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),
    ]
)


fcv_regression_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        ),
    ]
)


fcv_regression_linear_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                fcv_regression_numeric_transformer,
                fcv_numeric_features
            ),

            (
                "categorical",
                fcv_regression_categorical_transformer,
                fcv_categorical_features
            ),
        ]
    )
)

In [ ]:
# ================================================================
# 51.6 — TREE REGRESSION PREPROCESSOR
# ================================================================

fcv_regression_tree_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median",
                                add_indicator=True
                            )
                        )
                    ]
                ),
                fcv_numeric_features
            ),

            (
                "categorical",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),

                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False
                            )
                        ),
                    ]
                ),
                fcv_categorical_features
            ),
        ]
    )
)

### 51.7 — Economic baselines
We need strong non-ML benchmarks.

First:
> future 90-day revenue ≈ recent 90-day revenue

Second:
>future 90-day revenue ≈ half of historical 180-day revenue

In [ ]:
# ================================================================
# 51.7 — ECONOMIC BASELINES
# ================================================================

positive_validation_actual = (
    fcv_positive_validation_data[
        "fcv90"
    ].to_numpy()
)


recent90_revenue_baseline = (
    fcv_positive_validation_data[
        "r90_revenue"
    ]
    .to_numpy()
)


historical_run_rate_baseline = (
    0.50
    *
    fcv_positive_validation_data[
        "h180_revenue"
    ]
    .to_numpy()
)


train_positive_median = (
    fcv_positive_train_data[
        "fcv90"
    ].median()
)


median_value_baseline = np.full(
    len(
        positive_validation_actual
    ),
    train_positive_median
)

In [ ]:
# ================================================================
# 51.8 — VALUE CAPTURE FUNCTION
# ================================================================

def fcv_value_capture_at_fraction(
    actual_revenue,
    predicted_value,
    fraction
):

    evaluation = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    actual_revenue
                ),

            "predicted":
                np.asarray(
                    predicted_value
                ),
        }
    ).sort_values(
        "predicted",
        ascending=False,
        kind="mergesort"
    ).reset_index(drop=True)


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    total_actual = (
        evaluation[
            "actual"
        ].sum()
    )


    if total_actual <= 0:
        return np.nan


    return (
        evaluation
        .head(n_top)[
            "actual"
        ]
        .sum()
        /
        total_actual
    )

In [ ]:
# ================================================================
# 51.9 — POSITIVE FCV REGRESSION EVALUATION
# ================================================================

def evaluate_positive_fcv_model(
    model_name,
    y_true_revenue,
    predictions_revenue
):

    y_true_revenue = np.asarray(
        y_true_revenue
    )


    predictions_revenue = np.asarray(
        predictions_revenue
    )


    predictions_revenue = np.maximum(
        predictions_revenue,
        0
    )


    y_true_log = np.log1p(
        y_true_revenue
    )


    predicted_log = np.log1p(
        predictions_revenue
    )


    spearman_result = spearmanr(
        y_true_revenue,
        predictions_revenue,
        nan_policy="omit"
    )


    return pd.DataFrame(
        {
            "Model": [
                model_name
            ],

            "MAE": [
                mean_absolute_error(
                    y_true_revenue,
                    predictions_revenue
                )
            ],

            "RMSE": [
                np.sqrt(
                    mean_squared_error(
                        y_true_revenue,
                        predictions_revenue
                    )
                )
            ],

            "Log MAE": [
                mean_absolute_error(
                    y_true_log,
                    predicted_log
                )
            ],

            "Log RMSE": [
                np.sqrt(
                    mean_squared_error(
                        y_true_log,
                        predicted_log
                    )
                )
            ],

            "R2": [
                r2_score(
                    y_true_revenue,
                    predictions_revenue
                )
            ],

            "Spearman": [
                spearman_result.statistic
            ],

            "Capture @ 10%": [
                fcv_value_capture_at_fraction(
                    y_true_revenue,
                    predictions_revenue,
                    0.10
                )
            ],

            "Capture @ 20%": [
                fcv_value_capture_at_fraction(
                    y_true_revenue,
                    predictions_revenue,
                    0.20
                )
            ],

            "Capture @ 30%": [
                fcv_value_capture_at_fraction(
                    y_true_revenue,
                    predictions_revenue,
                    0.30
                )
            ],

            "Predicted Total / Actual Total": [
                predictions_revenue.sum()
                /
                y_true_revenue.sum()
            ],
        }
    )

In [ ]:
# ================================================================
# 51.10 — BASELINE PERFORMANCE
# ================================================================

fcv_median_baseline_results = (
    evaluate_positive_fcv_model(
        "Train Positive Median",
        positive_validation_actual,
        median_value_baseline
    )
)


fcv_recent90_baseline_results = (
    evaluate_positive_fcv_model(
        "Recent 90-Day Revenue",
        positive_validation_actual,
        recent90_revenue_baseline
    )
)


fcv_run_rate_baseline_results = (
    evaluate_positive_fcv_model(
        "180-Day Historical Run Rate",
        positive_validation_actual,
        historical_run_rate_baseline
    )
)


display(
    pd.concat(
        [
            fcv_median_baseline_results,
            fcv_recent90_baseline_results,
            fcv_run_rate_baseline_results,
        ],
        ignore_index=True
    )
)

In [ ]:
# ================================================================
# 51.11 — RIDGE LOG-VALUE MODEL
# ================================================================

fcv_positive_ridge = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_regression_linear_preprocessor
        ),

        (
            "model",
            Ridge(
                alpha=10.0
            )
        ),
    ]
)


fcv_positive_ridge.fit(
    X_fcv_positive_train,
    y_fcv_positive_train
)

In [ ]:
ridge_train_log_predictions = (
    fcv_positive_ridge.predict(
        X_fcv_positive_train
    )
)


ridge_validation_log_predictions = (
    fcv_positive_ridge.predict(
        X_fcv_positive_validation
    )
)

### 51.12 — Smearing correction
Exponentiating a log prediction directly tends to underestimate arithmetic mean revenue.

Use a training-only smearing factor.

In [ ]:
# ================================================================
# 51.12 — RIDGE SMEARING CORRECTION
# ================================================================

ridge_log_residuals = (
    y_fcv_positive_train.to_numpy()
    -
    ridge_train_log_predictions
)


ridge_smearing_factor = (
    np.exp(
        ridge_log_residuals
    )
    .mean()
)


ridge_validation_revenue_predictions = (
    np.exp(
        ridge_validation_log_predictions
    )
    *
    ridge_smearing_factor
    -
    1
)


ridge_validation_revenue_predictions = np.maximum(
    ridge_validation_revenue_predictions,
    0
)


print(
    f"Ridge smearing factor: "
    f"{ridge_smearing_factor:.4f}"
)

In [ ]:
fcv_ridge_results = (
    evaluate_positive_fcv_model(
        "Ridge Log-FCV90",
        positive_validation_actual,
        ridge_validation_revenue_predictions
    )
)


display(
    fcv_ridge_results
)

### 51.13 — Random Forest log-value regressor

In [ ]:
# ================================================================
# 51.13 — RANDOM FOREST REGRESSION
# ================================================================

fcv_positive_random_forest = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_regression_tree_preprocessor
        ),

        (
            "model",
            RandomForestRegressor(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=15,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


fcv_positive_random_forest.fit(
    X_fcv_positive_train,
    y_fcv_positive_train
)

### 51.14 — RF smearing

In [ ]:
# ================================================================
# 51.14 — RANDOM FOREST SMEARING
# ================================================================

rf_train_log_predictions = (
    fcv_positive_random_forest
    .predict(
        X_fcv_positive_train
    )
)


rf_validation_log_predictions = (
    fcv_positive_random_forest
    .predict(
        X_fcv_positive_validation
    )
)


rf_log_residuals = (
    y_fcv_positive_train.to_numpy()
    -
    rf_train_log_predictions
)


rf_smearing_factor = (
    np.exp(
        rf_log_residuals
    )
    .mean()
)


rf_validation_revenue_predictions = (
    np.exp(
        rf_validation_log_predictions
    )
    *
    rf_smearing_factor
    -
    1
)


rf_validation_revenue_predictions = np.maximum(
    rf_validation_revenue_predictions,
    0
)


print(
    f"RF smearing factor: "
    f"{rf_smearing_factor:.4f}"
)

In [ ]:
fcv_rf_regression_results = (
    evaluate_positive_fcv_model(
        "Random Forest Log-FCV90",
        positive_validation_actual,
        rf_validation_revenue_predictions
    )
)


display(
    fcv_rf_regression_results
)

### 51.15 — Gradient Boosting log-value model

In [ ]:
# ================================================================
# 51.15 — GRADIENT BOOSTING REGRESSION
# ================================================================

fcv_positive_gradient_boosting = Pipeline(
    steps=[
        (
            "preprocessing",
            fcv_regression_tree_preprocessor
        ),

        (
            "model",
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=20,
                loss="huber",
                random_state=42
            )
        ),
    ]
)


fcv_positive_gradient_boosting.fit(
    X_fcv_positive_train,
    y_fcv_positive_train
)

In [ ]:
# ================================================================
# 51.16 — GRADIENT BOOSTING SMEARING
# ================================================================

gb_reg_train_log_predictions = (
    fcv_positive_gradient_boosting
    .predict(
        X_fcv_positive_train
    )
)


gb_reg_validation_log_predictions = (
    fcv_positive_gradient_boosting
    .predict(
        X_fcv_positive_validation
    )
)


gb_reg_log_residuals = (
    y_fcv_positive_train.to_numpy()
    -
    gb_reg_train_log_predictions
)


gb_reg_smearing_factor = (
    np.exp(
        gb_reg_log_residuals
    )
    .mean()
)


gb_reg_validation_revenue_predictions = (
    np.exp(
        gb_reg_validation_log_predictions
    )
    *
    gb_reg_smearing_factor
    -
    1
)


gb_reg_validation_revenue_predictions = np.maximum(
    gb_reg_validation_revenue_predictions,
    0
)


print(
    f"GB smearing factor: "
    f"{gb_reg_smearing_factor:.4f}"
)

In [ ]:
fcv_gb_regression_results = (
    evaluate_positive_fcv_model(
        "Gradient Boosting Log-FCV90",
        positive_validation_actual,
        gb_reg_validation_revenue_predictions
    )
)


display(
    fcv_gb_regression_results
)

### 51.17 — Full Stage B comparison

In [ ]:
# ================================================================
# 51.17 — STAGE B MODEL COMPARISON
# ================================================================

stage_b_model_comparison = (
    pd.concat(
        [
            fcv_median_baseline_results,
            fcv_recent90_baseline_results,
            fcv_run_rate_baseline_results,
            fcv_ridge_results,
            fcv_rf_regression_results,
            fcv_gb_regression_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "Capture @ 20%",
            "Spearman",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    stage_b_model_comparison
)

In [ ]:
# ================================================================
# 51.18 — CONDITIONAL VALUE CALIBRATION
# ================================================================

def fcv_value_quintile_table(
    actual,
    predicted
):

    evaluation = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    actual
                ),

            "predicted":
                np.asarray(
                    predicted
                ),
        }
    )


    evaluation[
        "predicted_value_quintile"
    ] = pd.qcut(
        evaluation[
            "predicted"
        ]
        .rank(
            method="first"
        ),
        q=5,
        labels=[
            "Q1 Lowest",
            "Q2",
            "Q3",
            "Q4",
            "Q5 Highest",
        ]
    )


    return (
        evaluation
        .groupby(
            "predicted_value_quintile",
            observed=True,
            as_index=False
        )
        .agg(
            episodes=(
                "actual",
                "size"
            ),

            average_actual_fcv90=(
                "actual",
                "mean"
            ),

            median_actual_fcv90=(
                "actual",
                "median"
            ),

            average_predicted_fcv90=(
                "predicted",
                "mean"
            ),

            total_actual_fcv90=(
                "actual",
                "sum"
            ),

            total_predicted_fcv90=(
                "predicted",
                "sum"
            ),
        )
    )

In [ ]:
# ================================================================
# 51.19 — MODEL VALUE QUINTILES
# ================================================================

fcv_ridge_value_quintiles = (
    fcv_value_quintile_table(
        positive_validation_actual,
        ridge_validation_revenue_predictions
    )
)


fcv_rf_value_quintiles = (
    fcv_value_quintile_table(
        positive_validation_actual,
        rf_validation_revenue_predictions
    )
)


fcv_gb_value_quintiles = (
    fcv_value_quintile_table(
        positive_validation_actual,
        gb_reg_validation_revenue_predictions
    )
)


display(
    fcv_ridge_value_quintiles
)

display(
    fcv_rf_value_quintiles
)

display(
    fcv_gb_value_quintiles
)

In [ ]:
# ================================================================
# 51.20 — TOP VALUE CAPTURE
# ================================================================

stage_b_capture_summary = (
    stage_b_model_comparison[
        [
            "Model",
            "Capture @ 10%",
            "Capture @ 20%",
            "Capture @ 30%",
            "Spearman",
            "Predicted Total / Actual Total",
        ]
    ]
    .copy()
)


display(
    stage_b_capture_summary
)

In [ ]:
# ================================================================
# 51.20 — TOP VALUE CAPTURE
# ================================================================

stage_b_capture_summary = (
    stage_b_model_comparison[
        [
            "Model",
            "Capture @ 10%",
            "Capture @ 20%",
            "Capture @ 30%",
            "Spearman",
            "Predicted Total / Actual Total",
        ]
    ]
    .copy()
)


display(
    stage_b_capture_summary
)

### Step 51 Summary — Stage B Positive Future Value Regression

Step 51 developed the second component of the Future Customer Value hurdle architecture.

Stage B estimates:

> **E(FCV90 | FCV90 > 0)**

or the expected future 90-day revenue generated by a customer conditional on the customer generating positive future revenue.

### Positive Future Value Is Predictable

All serious models produced meaningful separation between customers with lower and higher future economic value.

Gradient Boosting achieved the strongest ranking performance with approximately:

- **Spearman correlation: 0.683**
- **Capture@10%: 49.0%**
- **Capture@20%: 60.7%**
- **Capture@30%: 69.2%**

The highest predicted-value quintile generated average actual FCV90 above **3,100**, compared with below **300** in the lowest quintile.

### Ridge Produced the Strongest Absolute Error Performance

Ridge Log-FCV90 achieved approximately:

- **MAE: 486**
- **Log MAE: 0.539**
- **Log RMSE: 0.706**
- **Spearman: 0.679**

Its value-ranking performance was very close to Gradient Boosting.

However, Ridge underpredicted aggregate Validation revenue by approximately **12%**.

### Gradient Boosting Produced the Strongest Economic Ranking

Gradient Boosting achieved the strongest top-customer revenue concentration.

However, its predicted conditional revenue total was approximately **14.5% above** the actual Validation total.

The model therefore ranks customers very effectively but requires scrutiny of temporal revenue calibration before final selection.

### Historical Revenue Remains an Exceptionally Strong Benchmark

A simple benchmark based on:

> **50% × prior 180-day customer revenue**

achieved approximately:

- **RMSE: 1,631**
- **R²: 0.748**
- **Predicted / Actual total: 0.961**

This benchmark outperformed the ML models on several raw revenue-error measures.

Historical customer economic scale is therefore a major driver of positive future value.

### Random Forest Was Not Competitive Enough

Random Forest did not lead on:

- ranking;
- absolute error;
- aggregate calibration;
- or revenue capture.

It will therefore not be carried forward as a primary Stage B candidate.

## Stage B Candidate Position

Three approaches remain:

1. **180-Day Historical Run Rate**
2. **Ridge Log-FCV90**
3. **Gradient Boosting Log-FCV90**

Final selection will depend on:

- purged temporal robustness;
- ranking stability;
- revenue capture;
- original-scale error;
- and aggregate revenue calibration.

The final FCV Test population remains unopened.


## Step 52 — Stage B Temporal Robustness and Model Selection

Step 52 evaluates the three strongest positive Future Customer Value approaches across purged historical periods.

The finalists are:

1. 180-Day Historical Run Rate;
2. Ridge Log-FCV90;
3. Gradient Boosting Log-FCV90.

Each historical fold includes only training episodes whose future 90-day outcome was fully known before the later validation prediction date.

For log-revenue models, the retransformation smearing factor is re-estimated exclusively from each fold's training population.

The models are compared on:

- MAE;
- RMSE;
- R²;
- Spearman rank correlation;
- future-revenue capture;
- and aggregate predicted-to-actual revenue.

The final FCV Test population remains unopened.

In [ ]:
# ================================================================
# 52.2 — GOVERNANCE
# ================================================================

assert FCV_TEST_SET_OPENED is False
assert FCV_STAGE_A_MODEL_FROZEN is True

print(
    "✓ Stage A remains frozen."
)

print(
    "✓ FCV Test population remains unopened."
)

In [ ]:
# ================================================================
# 52.3 — POSITIVE-VALUE PURGED FOLDS
# ================================================================

fcv_stage_b_purged_folds = []


for validation_period in (
    FCV_PURGED_VALIDATION_PERIODS
):

    validation_month_start = (
        validation_period
        .to_timestamp(
            how="start"
        )
    )


    train_mask = (
        (fcv_temporal_dataset["fcv90"] > 0)
        &
        (
            fcv_temporal_dataset[
                "outcome_end"
            ]
            <
            validation_month_start
        )
    )


    validation_mask = (
        (
            fcv_temporal_dataset[
                "prediction_period"
            ]
            ==
            validation_period
        )
        &
        (
            fcv_temporal_dataset[
                "fcv90"
            ]
            > 0
        )
    )


    fold_train = (
        fcv_temporal_dataset.loc[
            train_mask
        ]
        .copy()
    )


    fold_validation = (
        fcv_temporal_dataset.loc[
            validation_mask
        ]
        .copy()
    )


    if (
        len(fold_train) > 0
        and
        len(fold_validation) > 0
    ):

        fcv_stage_b_purged_folds.append(
            {
                "validation_period":
                    validation_period,

                "train":
                    fold_train,

                "validation":
                    fold_validation,
            }
        )


print(
    f"Stage B purged folds: "
    f"{len(fcv_stage_b_purged_folds)}"
)

In [ ]:
# ================================================================
# 52.4 — STAGE B FOLD AUDIT
# ================================================================

fcv_stage_b_fold_audit = []


for fold in fcv_stage_b_purged_folds:

    train = fold[
        "train"
    ]

    validation = fold[
        "validation"
    ]


    fcv_stage_b_fold_audit.append(
        {
            "validation_period":
                str(
                    fold[
                        "validation_period"
                    ]
                ),

            "train_positive_episodes":
                len(train),

            "validation_positive_episodes":
                len(validation),

            "latest_training_prediction":
                train[
                    "prediction_date"
                ]
                .max(),

            "latest_training_outcome_known":
                train[
                    "outcome_end"
                ]
                .max(),

            "validation_prediction_date":
                validation[
                    "prediction_date"
                ]
                .min(),

            "validation_mean_fcv90":
                validation[
                    "fcv90"
                ]
                .mean(),

            "validation_median_fcv90":
                validation[
                    "fcv90"
                ]
                .median(),
        }
    )


fcv_stage_b_fold_audit = pd.DataFrame(
    fcv_stage_b_fold_audit
)


display(
    fcv_stage_b_fold_audit
)

In [ ]:
# ================================================================
# 52.5 — STAGE B PURGED TEMPORAL EVALUATION
# ================================================================

fcv_stage_b_temporal_records = []


for fold_number, fold in enumerate(
    fcv_stage_b_purged_folds,
    start=1
):

    train = fold[
        "train"
    ]

    validation = fold[
        "validation"
    ]


    X_train_fold = (
        train[
            fcv_model_features
        ]
    )


    y_train_revenue = (
        train[
            "fcv90"
        ]
        .to_numpy()
    )


    y_train_log = np.log1p(
        y_train_revenue
    )


    X_validation_fold = (
        validation[
            fcv_model_features
        ]
    )


    y_validation_revenue = (
        validation[
            "fcv90"
        ]
        .to_numpy()
    )


    # ------------------------------------------------------------
    # 1. HISTORICAL RUN-RATE BASELINE
    # ------------------------------------------------------------

    run_rate_predictions = (
        0.50
        *
        validation[
            "h180_revenue"
        ]
        .to_numpy()
    )


    run_rate_result = (
        evaluate_positive_fcv_model(
            "180-Day Historical Run Rate",
            y_validation_revenue,
            run_rate_predictions
        )
        .iloc[0]
        .to_dict()
    )


    # ------------------------------------------------------------
    # 2. RIDGE
    # ------------------------------------------------------------

    fold_ridge = clone(
        fcv_positive_ridge
    )


    fold_ridge.fit(
        X_train_fold,
        y_train_log
    )


    ridge_train_log = (
        fold_ridge.predict(
            X_train_fold
        )
    )


    ridge_validation_log = (
        fold_ridge.predict(
            X_validation_fold
        )
    )


    ridge_fold_residuals = (
        y_train_log
        -
        ridge_train_log
    )


    ridge_fold_smearing = (
        np.exp(
            ridge_fold_residuals
        )
        .mean()
    )


    ridge_fold_predictions = (
        np.exp(
            ridge_validation_log
        )
        *
        ridge_fold_smearing
        -
        1
    )


    ridge_fold_predictions = np.maximum(
        ridge_fold_predictions,
        0
    )


    ridge_result = (
        evaluate_positive_fcv_model(
            "Ridge Log-FCV90",
            y_validation_revenue,
            ridge_fold_predictions
        )
        .iloc[0]
        .to_dict()
    )


    # ------------------------------------------------------------
    # 3. GRADIENT BOOSTING
    # ------------------------------------------------------------

    fold_gb = clone(
        fcv_positive_gradient_boosting
    )


    fold_gb.fit(
        X_train_fold,
        y_train_log
    )


    gb_train_log = (
        fold_gb.predict(
            X_train_fold
        )
    )


    gb_validation_log = (
        fold_gb.predict(
            X_validation_fold
        )
    )


    gb_fold_residuals = (
        y_train_log
        -
        gb_train_log
    )


    gb_fold_smearing = (
        np.exp(
            gb_fold_residuals
        )
        .mean()
    )


    gb_fold_predictions = (
        np.exp(
            gb_validation_log
        )
        *
        gb_fold_smearing
        -
        1
    )


    gb_fold_predictions = np.maximum(
        gb_fold_predictions,
        0
    )


    gb_result = (
        evaluate_positive_fcv_model(
            "Gradient Boosting Log-FCV90",
            y_validation_revenue,
            gb_fold_predictions
        )
        .iloc[0]
        .to_dict()
    )


    # ------------------------------------------------------------
    # RECORD
    # ------------------------------------------------------------

    for model_result in [
        run_rate_result,
        ridge_result,
        gb_result,
    ]:

        model_result[
            "fold"
        ] = fold_number

        model_result[
            "validation_period"
        ] = str(
            fold[
                "validation_period"
            ]
        )

        model_result[
            "train_positive_episodes"
        ] = len(
            train
        )

        model_result[
            "validation_positive_episodes"
        ] = len(
            validation
        )


        fcv_stage_b_temporal_records.append(
            model_result
        )


fcv_stage_b_temporal_results = pd.DataFrame(
    fcv_stage_b_temporal_records
)


display(
    fcv_stage_b_temporal_results
)

In [ ]:
# ================================================================
# 52.6 — STAGE B TEMPORAL SUMMARY
# ================================================================

fcv_stage_b_temporal_summary = (
    fcv_stage_b_temporal_results
    .groupby(
        "Model",
        as_index=False
    )
    .agg(
        folds=(
            "fold",
            "nunique"
        ),

        mean_MAE=(
            "MAE",
            "mean"
        ),

        mean_RMSE=(
            "RMSE",
            "mean"
        ),

        mean_Log_MAE=(
            "Log MAE",
            "mean"
        ),

        mean_Log_RMSE=(
            "Log RMSE",
            "mean"
        ),

        mean_R2=(
            "R2",
            "mean"
        ),

        mean_Spearman=(
            "Spearman",
            "mean"
        ),

        min_Spearman=(
            "Spearman",
            "min"
        ),

        mean_Capture_10=(
            "Capture @ 10%",
            "mean"
        ),

        mean_Capture_20=(
            "Capture @ 20%",
            "mean"
        ),

        mean_Capture_30=(
            "Capture @ 30%",
            "mean"
        ),

        mean_Predicted_Total_Ratio=(
            "Predicted Total / Actual Total",
            "mean"
        ),

        min_Predicted_Total_Ratio=(
            "Predicted Total / Actual Total",
            "min"
        ),

        max_Predicted_Total_Ratio=(
            "Predicted Total / Actual Total",
            "max"
        ),
    )
)


display(
    fcv_stage_b_temporal_summary
)

In [ ]:
# ================================================================
# 52.7 — STAGE B FOLD WINNERS
# ================================================================

fcv_stage_b_spearman_winners = (
    fcv_stage_b_temporal_results.loc[
        fcv_stage_b_temporal_results
        .groupby(
            "validation_period"
        )[
            "Spearman"
        ]
        .idxmax(),
        [
            "validation_period",
            "Model",
            "Spearman",
        ]
    ]
)


fcv_stage_b_capture20_winners = (
    fcv_stage_b_temporal_results.loc[
        fcv_stage_b_temporal_results
        .groupby(
            "validation_period"
        )[
            "Capture @ 20%"
        ]
        .idxmax(),
        [
            "validation_period",
            "Model",
            "Capture @ 20%",
        ]
    ]
)


fcv_stage_b_rmse_winners = (
    fcv_stage_b_temporal_results.loc[
        fcv_stage_b_temporal_results
        .groupby(
            "validation_period"
        )[
            "RMSE"
        ]
        .idxmin(),
        [
            "validation_period",
            "Model",
            "RMSE",
        ]
    ]
)


print(
    "Spearman winners:"
)

display(
    fcv_stage_b_spearman_winners
)


print(
    "Capture@20 winners:"
)

display(
    fcv_stage_b_capture20_winners
)


print(
    "RMSE winners:"
)

display(
    fcv_stage_b_rmse_winners
)

In [ ]:
# ================================================================
# 52.8 — AGGREGATE REVENUE CALIBRATION
# ================================================================

fcv_stage_b_temporal_results[
    "Aggregate Calibration Error"
] = (
    (
        fcv_stage_b_temporal_results[
            "Predicted Total / Actual Total"
        ]
        -
        1
    )
    .abs()
)


fcv_stage_b_aggregate_calibration = (
    fcv_stage_b_temporal_results
    .groupby(
        "Model",
        as_index=False
    )
    .agg(
        mean_absolute_aggregate_error=(
            "Aggregate Calibration Error",
            "mean"
        ),

        max_absolute_aggregate_error=(
            "Aggregate Calibration Error",
            "max"
        ),
    )
)


display(
    fcv_stage_b_aggregate_calibration
)

In [ ]:
# ================================================================
# 52.9 — STAGE B PRE-TEST SCORECARD
# ================================================================

fcv_stage_b_pretest_scorecard = pd.DataFrame(
    [
        {
            "Model":
                "180-Day Historical Run Rate",

            "Validation MAE":
                fcv_run_rate_baseline_results[
                    "MAE"
                ].iloc[0],

            "Validation RMSE":
                fcv_run_rate_baseline_results[
                    "RMSE"
                ].iloc[0],

            "Validation Spearman":
                fcv_run_rate_baseline_results[
                    "Spearman"
                ].iloc[0],

            "Validation Capture@20":
                fcv_run_rate_baseline_results[
                    "Capture @ 20%"
                ].iloc[0],

            "Validation Total Ratio":
                fcv_run_rate_baseline_results[
                    "Predicted Total / Actual Total"
                ].iloc[0],
        },

        {
            "Model":
                "Ridge Log-FCV90",

            "Validation MAE":
                fcv_ridge_results[
                    "MAE"
                ].iloc[0],

            "Validation RMSE":
                fcv_ridge_results[
                    "RMSE"
                ].iloc[0],

            "Validation Spearman":
                fcv_ridge_results[
                    "Spearman"
                ].iloc[0],

            "Validation Capture@20":
                fcv_ridge_results[
                    "Capture @ 20%"
                ].iloc[0],

            "Validation Total Ratio":
                fcv_ridge_results[
                    "Predicted Total / Actual Total"
                ].iloc[0],
        },

        {
            "Model":
                "Gradient Boosting Log-FCV90",

            "Validation MAE":
                fcv_gb_regression_results[
                    "MAE"
                ].iloc[0],

            "Validation RMSE":
                fcv_gb_regression_results[
                    "RMSE"
                ].iloc[0],

            "Validation Spearman":
                fcv_gb_regression_results[
                    "Spearman"
                ].iloc[0],

            "Validation Capture@20":
                fcv_gb_regression_results[
                    "Capture @ 20%"
                ].iloc[0],

            "Validation Total Ratio":
                fcv_gb_regression_results[
                    "Predicted Total / Actual Total"
                ].iloc[0],
        },
    ]
)


fcv_stage_b_pretest_scorecard = (
    fcv_stage_b_pretest_scorecard
    .merge(
        fcv_stage_b_temporal_summary[
            [
                "Model",
                "mean_MAE",
                "mean_RMSE",
                "mean_Spearman",
                "mean_Capture_20",
                "mean_Predicted_Total_Ratio",
            ]
        ],
        on="Model",
        how="left",
        validate="one_to_one"
    )
)


display(
    fcv_stage_b_pretest_scorecard
)

In [ ]:
# ================================================================
# 52.10 — STAGE B DECISION PENDING
# ================================================================

FCV_STAGE_B_MODEL_FROZEN = False

FINAL_FCV_STAGE_B_MODEL_NAME = None
FINAL_FCV_STAGE_B_MODEL = None


print(
    "Stage B remains unfrozen pending "
    "purged temporal evidence review."
)

In [ ]:
# ================================================================
# 52.11 — STEP 52 COMPLETE
# ================================================================

assert FCV_TEST_SET_OPENED is False
assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is False


print(
    "✓ Step 52 complete."
)

print(
    "✓ Stage B purged temporal validation completed."
)

print(
    "✓ Fold-specific smearing corrections used."
)

print(
    "✓ Ranking, revenue capture and aggregate calibration evaluated."
)

print(
    "✓ FCV final Test remains unopened."
)

### Step 52 Summary — Stage B Temporal Robustness

Step 52 evaluated the leading positive-FCV90 models using purged historical validation.

Each historical training population contained only positive-revenue episodes whose future 90-day outcomes had fully matured before the subsequent validation prediction point.

Fold-specific smearing corrections were estimated from training data only.

### Ridge Failed Temporal Robustness

Ridge Log-FCV90 performed strongly on the designated Validation population but became highly unstable across historical folds.

Across the purged folds it produced approximately:

- **mean MAE: 4,939**
- **mean RMSE: 146,260**
- **mean predicted/actual total ratio: 4.48×**

In its most extreme historical period, predicted aggregate conditional revenue approached approximately **8× actual revenue**.

This instability is unacceptable for a future-value system.

Ridge is therefore removed from Stage B contention.

### Historical Run Rate Remained an Exceptional Benchmark

The simple conditional-value benchmark:

> **50% × prior 180-day customer revenue**

achieved approximately:

- **mean MAE: 684**
- **mean RMSE: 2,470**
- **mean R²: 0.497**
- **mean Spearman: 0.605**
- **mean Capture@20%: 57.9%**

It achieved the lowest RMSE in all three historical folds.

Historical customer revenue scale therefore remains a highly robust predictor of future positive revenue magnitude.

### Gradient Boosting Produced Stronger Economic Ranking

Gradient Boosting achieved approximately:

- **mean Spearman: 0.617**
- **minimum Spearman: 0.601**
- **mean Capture@10%: 46.1%**
- **mean Capture@20%: 58.3%**
- **mean Capture@30%: 66.5%**

It produced the highest Spearman correlation in two of three historical folds and the highest Capture@20% in two of three folds.

Gradient Boosting therefore provides somewhat stronger customer-value ranking.

### Aggregate Revenue Calibration Was Mixed

The historical run-rate benchmark produced an average predicted/actual revenue ratio of approximately **0.88**.

Gradient Boosting produced an average ratio of approximately **1.02**.

However, both models showed meaningful variation between historical periods.

Mean absolute aggregate calibration error was approximately:

- **22.2% — Historical Run Rate**
- **23.8% — Gradient Boosting**

Neither candidate therefore dominates on aggregate calibration.

## Stage B Candidate Decision

Ridge is eliminated.

Two Stage B candidates remain:

1. **180-Day Historical Run Rate**
2. **Gradient Boosting Log-FCV90**

Stage B will not yet be frozen.

The final decision should be made in the context of the complete hurdle system:

> **Expected FCV90 = P(Future Active) × Conditional Positive FCV90**

The preferred Stage B specification will therefore be the one that produces the stronger complete Expected FCV90 model on the untouched Validation population.

The final Test population remains unopened.


## Step 53 — Combined Expected FCV90 Architecture

The Future Customer Value system contains two components.

### Stage A

Estimate:

> **P(Future Active)**

using the frozen Random Forest classifier.

### Stage B

Estimate:

> **E(FCV90 | Future Active)**

using a conditional positive-revenue model.

Two Stage B candidates remain:

1. 180-Day Historical Run Rate;
2. Gradient Boosting Log-FCV90.

The final expected-value estimate is:

> **Expected FCV90 = P(Future Active) × Conditional Positive FCV90**

Step 53 compares both complete hurdle architectures on the designated Validation population.

This is the final model-selection stage before the complete FCV architecture is frozen.

The final Test population remains unopened.

In [ ]:
# ================================================================
# 53.2 — GOVERNANCE
# ================================================================

assert FCV_TEST_SET_OPENED is False
assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is False


print(
    "✓ Stage A is frozen."
)

print(
    "✓ Stage B remains under final comparison."
)

print(
    "✓ FCV Test remains unopened."
)

In [ ]:
# ================================================================
# 53.3 — FROZEN STAGE A VALIDATION PROBABILITIES
# ================================================================

fcv_validation_activity_probability = (
    validation_fcv_rf_probabilities.copy()
)


assert (
    len(
        fcv_validation_activity_probability
    )
    ==
    len(
        fcv_validation_data
    )
)


print(
    "✓ Frozen Stage A probabilities prepared."
)

In [ ]:
# ================================================================
# 53.4 — RUN-RATE CONDITIONAL VALUE
# ================================================================

validation_conditional_value_run_rate = (
    0.50
    *
    fcv_validation_data[
        "h180_revenue"
    ]
    .to_numpy()
)


print(
    "Run-rate conditional predictions prepared."
)

### 53.5 — Candidate B conditional Gradient Boosting value
Important: Stage B was trained only on positive outcomes, but at scoring time we apply the fitted conditional-value function to every eligible customer.

In [ ]:
# ================================================================
# 53.5 — GB CONDITIONAL VALUE FOR ALL VALIDATION EPISODES
# ================================================================

gb_all_validation_log_predictions = (
    fcv_positive_gradient_boosting
    .predict(
        X_fcv_validation
    )
)


validation_conditional_value_gb = (
    np.exp(
        gb_all_validation_log_predictions
    )
    *
    gb_reg_smearing_factor
    -
    1
)


validation_conditional_value_gb = np.maximum(
    validation_conditional_value_gb,
    0
)


print(
    "Gradient Boosting conditional predictions prepared "
    "for all Validation episodes."
)

In [ ]:
# ================================================================
# 53.6 — EXPECTED FCV90
# ================================================================

expected_fcv90_run_rate = (
    fcv_validation_activity_probability
    *
    validation_conditional_value_run_rate
)


expected_fcv90_gb = (
    fcv_validation_activity_probability
    *
    validation_conditional_value_gb
)


print(
    "✓ Both hurdle-model Expected FCV90 predictions created."
)

In [ ]:
# ================================================================
# 53.7 — ACTUAL FCV90
# ================================================================

actual_validation_fcv90 = (
    fcv_validation_data[
        "fcv90"
    ]
    .to_numpy()
)


assert (
    len(
        actual_validation_fcv90
    )
    ==
    len(
        expected_fcv90_gb
    )
)

### 53.8 — Complete hurdle evaluation function
Our primary aim is economic ranking, not exact point forecasting alone.

In [ ]:
# ================================================================
# 53.8 — COMPLETE FCV EVALUATION
# ================================================================

def evaluate_complete_fcv_model(
    model_name,
    actual,
    predicted
):

    actual = np.asarray(
        actual
    )


    predicted = np.maximum(
        np.asarray(
            predicted
        ),
        0
    )


    spearman_result = spearmanr(
        actual,
        predicted,
        nan_policy="omit"
    )


    return pd.DataFrame(
        {
            "Model": [
                model_name
            ],

            "MAE": [
                mean_absolute_error(
                    actual,
                    predicted
                )
            ],

            "RMSE": [
                np.sqrt(
                    mean_squared_error(
                        actual,
                        predicted
                    )
                )
            ],

            "R2": [
                r2_score(
                    actual,
                    predicted
                )
            ],

            "Spearman": [
                spearman_result.statistic
            ],

            "Capture @ 1%": [
                fcv_value_capture_at_fraction(
                    actual,
                    predicted,
                    0.01
                )
            ],

            "Capture @ 5%": [
                fcv_value_capture_at_fraction(
                    actual,
                    predicted,
                    0.05
                )
            ],

            "Capture @ 10%": [
                fcv_value_capture_at_fraction(
                    actual,
                    predicted,
                    0.10
                )
            ],

            "Capture @ 20%": [
                fcv_value_capture_at_fraction(
                    actual,
                    predicted,
                    0.20
                )
            ],

            "Capture @ 30%": [
                fcv_value_capture_at_fraction(
                    actual,
                    predicted,
                    0.30
                )
            ],

            "Predicted Total": [
                predicted.sum()
            ],

            "Actual Total": [
                actual.sum()
            ],

            "Predicted / Actual Total": [
                predicted.sum()
                /
                actual.sum()
            ],
        }
    )

In [ ]:
# ================================================================
# 53.9 — COMPLETE HURDLE COMPARISON
# ================================================================

fcv_hurdle_run_rate_results = (
    evaluate_complete_fcv_model(
        model_name=(
            "RF Activity × 180-Day Run Rate"
        ),
        actual=actual_validation_fcv90,
        predicted=expected_fcv90_run_rate
    )
)


fcv_hurdle_gb_results = (
    evaluate_complete_fcv_model(
        model_name=(
            "RF Activity × Gradient Boosting Value"
        ),
        actual=actual_validation_fcv90,
        predicted=expected_fcv90_gb
    )
)


fcv_complete_hurdle_comparison = (
    pd.concat(
        [
            fcv_hurdle_run_rate_results,
            fcv_hurdle_gb_results,
        ],
        ignore_index=True
    )
)


display(
    fcv_complete_hurdle_comparison
)

### 53.10 — Important single-stage benchmarks
We should compare the hurdle system with simple economic scores too.

In [ ]:
# ================================================================
# 53.10 — SIMPLE COMPLETE-VALUE BENCHMARKS
# ================================================================

validation_h180_half = (
    0.50
    *
    fcv_validation_data[
        "h180_revenue"
    ]
    .to_numpy()
)


validation_r90_value = (
    fcv_validation_data[
        "r90_revenue"
    ]
    .to_numpy()
)


fcv_simple_run_rate_complete = (
    evaluate_complete_fcv_model(
        "Raw 180-Day Run Rate",
        actual_validation_fcv90,
        validation_h180_half
    )
)


fcv_simple_recent90_complete = (
    evaluate_complete_fcv_model(
        "Raw Recent 90-Day Revenue",
        actual_validation_fcv90,
        validation_r90_value
    )
)

In [ ]:
fcv_complete_model_comparison = (
    pd.concat(
        [
            fcv_simple_run_rate_complete,
            fcv_simple_recent90_complete,
            fcv_hurdle_run_rate_results,
            fcv_hurdle_gb_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "Capture @ 20%",
            "Spearman",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    fcv_complete_model_comparison
)

In [ ]:
# ================================================================
# 53.11 — HURDLE VALUE-ADD
# ================================================================

raw_run_rate_capture20 = (
    fcv_simple_run_rate_complete[
        "Capture @ 20%"
    ].iloc[0]
)


fcv_hurdle_value_add = pd.DataFrame(
    {
        "Model": [
            "RF Activity × Run Rate",
            "RF Activity × GB Value",
        ],

        "Capture@20": [
            fcv_hurdle_run_rate_results[
                "Capture @ 20%"
            ].iloc[0],

            fcv_hurdle_gb_results[
                "Capture @ 20%"
            ].iloc[0],
        ],
    }
)


fcv_hurdle_value_add[
    "Capture@20 Advantage vs Raw Run Rate"
] = (
    fcv_hurdle_value_add[
        "Capture@20"
    ]
    -
    raw_run_rate_capture20
)


display(
    fcv_hurdle_value_add
)

In [ ]:
# ================================================================
# 53.12 — COMPLETE FCV DECILE FUNCTION
# ================================================================

def create_complete_fcv_deciles(
    actual,
    predicted
):

    evaluation = pd.DataFrame(
        {
            "actual_fcv90":
                np.asarray(
                    actual
                ),

            "predicted_fcv90":
                np.asarray(
                    predicted
                ),
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "predicted_fcv90",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(drop=True)
    )


    evaluation[
        "value_decile"
    ] = (
        pd.qcut(
            np.arange(
                len(evaluation)
            ),
            q=10,
            labels=False
        )
        +
        1
    )


    return (
        evaluation
        .groupby(
            "value_decile",
            as_index=False
        )
        .agg(
            episodes=(
                "actual_fcv90",
                "size"
            ),

            average_actual_fcv90=(
                "actual_fcv90",
                "mean"
            ),

            median_actual_fcv90=(
                "actual_fcv90",
                "median"
            ),

            average_predicted_fcv90=(
                "predicted_fcv90",
                "mean"
            ),

            total_actual_fcv90=(
                "actual_fcv90",
                "sum"
            ),

            total_predicted_fcv90=(
                "predicted_fcv90",
                "sum"
            ),
        )
    )

In [ ]:
fcv_hurdle_run_rate_deciles = (
    create_complete_fcv_deciles(
        actual_validation_fcv90,
        expected_fcv90_run_rate
    )
)


fcv_hurdle_gb_deciles = (
    create_complete_fcv_deciles(
        actual_validation_fcv90,
        expected_fcv90_gb
    )
)


display(
    fcv_hurdle_run_rate_deciles
)


display(
    fcv_hurdle_gb_deciles
)

In [ ]:
# ================================================================
# 53.13 — ECONOMIC CAPTURE CURVE
# ================================================================

complete_fcv_capture_records = []


complete_fcv_scores = {
    "Raw 180-Day Run Rate":
        validation_h180_half,

    "RF Activity × Run Rate":
        expected_fcv90_run_rate,

    "RF Activity × GB Value":
        expected_fcv90_gb,
}


for model_name, score in (
    complete_fcv_scores.items()
):

    for fraction in [
        0.01,
        0.05,
        0.10,
        0.20,
        0.30,
    ]:

        capture = (
            fcv_value_capture_at_fraction(
                actual_validation_fcv90,
                score,
                fraction
            )
        )


        complete_fcv_capture_records.append(
            {
                "Model":
                    model_name,

                "Customer Fraction":
                    fraction,

                "Future Revenue Capture":
                    capture,

                "Future Revenue Capture (%)":
                    capture
                    *
                    100,
            }
        )


complete_fcv_capture_curve = pd.DataFrame(
    complete_fcv_capture_records
)


display(
    complete_fcv_capture_curve
)

In [ ]:
plt.figure(
    figsize=(10, 6)
)


sns.lineplot(
    data=complete_fcv_capture_curve,
    x="Customer Fraction",
    y="Future Revenue Capture",
    hue="Model",
    marker="o"
)


plt.plot(
    [0, 0.30],
    [0, 0.30],
    linestyle="--",
    label="Random targeting"
)


plt.title(
    "Validation Future Revenue Capture"
)

plt.xlabel(
    "Customer Fraction Targeted"
)

plt.ylabel(
    "Share of Future Revenue Captured"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 53.14 — COMPLETE SCORE RELATIONSHIPS
# ================================================================

fcv_complete_score_correlations = (
    pd.DataFrame(
        {
            "Actual_FCV90":
                actual_validation_fcv90,

            "Historical_Run_Rate":
                validation_h180_half,

            "Hurdle_Run_Rate":
                expected_fcv90_run_rate,

            "Hurdle_GB":
                expected_fcv90_gb,
        }
    )
    .corr(
        method="spearman"
    )
)


display(
    fcv_complete_score_correlations.round(3)
)

In [ ]:
# ================================================================
# 53.15 — COMPLETE FCV AGGREGATE CALIBRATION
# ================================================================

fcv_complete_aggregate_calibration = pd.DataFrame(
    {
        "Model": [
            "Raw 180-Day Run Rate",
            "RF Activity × Run Rate",
            "RF Activity × GB Value",
        ],

        "Predicted Total": [
            validation_h180_half.sum(),
            expected_fcv90_run_rate.sum(),
            expected_fcv90_gb.sum(),
        ],

        "Actual Total": [
            actual_validation_fcv90.sum(),
            actual_validation_fcv90.sum(),
            actual_validation_fcv90.sum(),
        ],
    }
)


fcv_complete_aggregate_calibration[
    "Predicted / Actual"
] = (
    fcv_complete_aggregate_calibration[
        "Predicted Total"
    ]
    /
    fcv_complete_aggregate_calibration[
        "Actual Total"
    ]
)


fcv_complete_aggregate_calibration[
    "Absolute Aggregate Error (%)"
] = (
    (
        fcv_complete_aggregate_calibration[
            "Predicted / Actual"
        ]
        -
        1
    )
    .abs()
    *
    100
)


display(
    fcv_complete_aggregate_calibration
)

In [ ]:
# ================================================================
# 53.16 — FINAL ARCHITECTURE STILL UNFROZEN
# ================================================================

FCV_STAGE_B_MODEL_FROZEN = False
FCV_COMPLETE_MODEL_FROZEN = False


print(
    "Stage B and complete FCV architecture remain "
    "unfrozen pending Step 53 evidence review."
)

print(
    "Final Test remains unopened."
)

In [ ]:
# ================================================================
# 53.17 — STEP 53 COMPLETION
# ================================================================

assert FCV_TEST_SET_OPENED is False
assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is False


print(
    "✓ Step 53 complete."
)

print(
    "✓ Frozen Stage A integrated."
)

print(
    "✓ Both Stage B finalists tested in the complete hurdle model."
)

print(
    "✓ Simple economic benchmarks retained."
)

print(
    "✓ Complete future-revenue ranking evaluated."
)

print(
    "✓ Final FCV Test remains unopened."
)

### Step 53 Summary — Final Combined FCV Architecture

Step 53 combined the frozen Future Activity model with the two remaining conditional-value candidates.

The complete Future Customer Value architecture is:

> **Expected FCV90 = P(Future Active) × Conditional Positive FCV90**

Two hurdle architectures were evaluated:

1. Random Forest Activity × 180-Day Historical Run Rate;
2. Random Forest Activity × Gradient Boosting Conditional Value.

### The Hurdle Architecture Substantially Improves Historical Revenue Alone

Using raw historical revenue scale without an activity probability produced substantial aggregate overprediction.

The raw 180-day run-rate benchmark predicted approximately **34.2% more future revenue than actually occurred**.

After multiplying by the frozen Random Forest future-activity probability, aggregate error declined to approximately:

> **2.66%**

This confirms the economic value of the hurdle architecture.

### Random Forest Activity × Run Rate Produced the Strongest Overall Forecast

The complete Run-Rate hurdle achieved approximately:

- **MAE: 349**
- **RMSE: 1,210**
- **R²: 0.712**
- **Spearman: 0.502**
- **Capture@10%: 58.4%**
- **Capture@20%: 71.4%**
- **Capture@30%: 79.1%**
- **Predicted / Actual Revenue: 0.973**

The model therefore performed strongly on both individual customer ranking and aggregate revenue estimation.

### Gradient Boosting Added Only Marginal Capture Improvement

The Gradient Boosting hurdle achieved slightly higher future-revenue capture at several customer cutoffs.

For example:

- Capture@20% was approximately **71.42%** compared with **71.36%** for the Run-Rate hurdle.

This difference is economically negligible.

However, the Gradient Boosting hurdle overpredicted aggregate Validation FCV90 by approximately **18.0%**, compared with only **2.7% aggregate error** for the Run-Rate hurdle.

It also produced weaker MAE, RMSE, R² and overall Spearman ranking.

### Final Stage B Decision

The frozen Stage B specification is therefore:

> **50% × Prior 180-Day Customer Revenue**

This simple structural-value model is preferred because it provides:

- stronger original-scale forecast accuracy;
- substantially better aggregate revenue calibration;
- strong customer ranking;
- strong revenue concentration;
- greater temporal robustness;
- and materially lower model complexity.

### Final FCV Architecture

The complete frozen Future Customer Value model is:

> **Expected FCV90 = P(Future Active from Random Forest) × 50% × Prior 180-Day Revenue**

This architecture separates:

- the probability that a customer will become economically active;
- from the structural revenue scale associated with that customer.

The final FCV Test population remains unopened.

## STEP 54 — FREEZE THE COMPLETE FCV MODEL
### 54.1 — Freeze Stage B


In [ ]:
# ================================================================
# 54.1 — FREEZE STAGE B
# ================================================================

FINAL_FCV_STAGE_B_MODEL_NAME = (
    "180-Day Historical Run Rate"
)

FINAL_FCV_STAGE_B_MODEL = (
    "0.50 × h180_revenue"
)

FCV_STAGE_B_MODEL_FROZEN = True


print(
    f"Frozen Stage B model: "
    f"{FINAL_FCV_STAGE_B_MODEL_NAME}"
)

print(
    "✓ Stage B specification frozen."
)

In [ ]:
# ================================================================
# 54.2 — FREEZE COMPLETE FCV ARCHITECTURE
# ================================================================

FINAL_FCV_MODEL_NAME = (
    "Random Forest Activity × 180-Day Historical Run Rate"
)


FINAL_FCV_FORMULA = (
    "P(Future Active) × 0.50 × h180_revenue"
)


FCV_COMPLETE_MODEL_FROZEN = True


print(
    f"Final FCV architecture: "
    f"{FINAL_FCV_MODEL_NAME}"
)

print(
    f"Formula: "
    f"{FINAL_FCV_FORMULA}"
)

print(
    "✓ Complete FCV architecture frozen."
)

In [ ]:
# ================================================================
# 54.3 — FINAL FCV SPECIFICATION
# ================================================================

final_fcv_specification = pd.DataFrame(
    {
        "Component": [
            "Business target",
            "Prediction horizon",
            "Eligibility window",
            "Stage A target",
            "Stage A model",
            "Stage A trees",
            "Stage A max depth",
            "Stage A minimum leaf",
            "Stage B target",
            "Stage B model",
            "Complete formula",
            "Primary ranking metric",
            "Primary economic metric",
            "Validation design",
            "Validation aggregate error",
            "Test status",
        ],

        "Frozen Specification": [
            "Future 90-Day Customer Value",
            "90 days",
            "At least one qualifying purchase in prior 180 days",
            "P(FCV90 > 0)",
            "Random Forest Classifier",
            500,
            10,
            20,
            "Conditional positive FCV90",
            "50% × prior 180-day revenue",
            "P(Future Active) × 50% × prior 180-day revenue",
            "Spearman / revenue ranking",
            "Future revenue capture",
            "Purged chronological validation",
            "2.66%",
            "UNOPENED",
        ],
    }
)


display(
    final_fcv_specification
)

In [ ]:
# ================================================================
# 54.4 — PRE-TEST GOVERNANCE LOCK
# ================================================================

assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is True
assert FCV_COMPLETE_MODEL_FROZEN is True
assert FCV_TEST_SET_OPENED is False


print(
    "✓ Stage A frozen."
)

print(
    "✓ Stage B frozen."
)

print(
    "✓ Complete hurdle architecture frozen."
)

print(
    "✓ No further Validation-driven model changes permitted."
)

print(
    "✓ Final FCV Test remains unopened."
)

## Step 55 — Final Untouched FCV Test

All Future Customer Value modelling decisions are now frozen.

The final architecture is:

> **Expected FCV90 = P(Future Active) × 50% × Prior 180-Day Revenue**

Stage A uses the frozen Random Forest classifier.

Stage B uses the frozen 180-day historical run-rate specification.

The Test population covers July and August 2011.

To preserve exact historical information availability, the Stage A classifier will be refitted separately for each Test prediction month using only episodes whose 90-day target had fully matured before that month's prediction date.

The Test outcomes will then be opened exactly once for final evaluation.

No model architecture, feature, hyperparameter or formula may be changed after Test results are observed.

In [ ]:
# ================================================================
# 55.2 — TEST PREDICTION DATES
# ================================================================

fcv_test_prediction_dates = (
    fcv_test_data[
        "prediction_date"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


print(
    "Final Test prediction dates:"
)

for date in fcv_test_prediction_dates:
    print(
        pd.Timestamp(date).date()
    )

In [ ]:
# ================================================================
# 55.3 — ROLLING OUT-OF-TIME TEST PREDICTIONS
# ================================================================

final_fcv_test_prediction_frames = []


for test_prediction_date in (
    fcv_test_prediction_dates
):

    test_prediction_date = pd.Timestamp(
        test_prediction_date
    )


    # ------------------------------------------------------------
    # Development labels must be fully known before prediction date
    # ------------------------------------------------------------

    development_data = (
        fcv_model_dataset.loc[
            fcv_model_dataset[
                "outcome_end"
            ]
            <
            test_prediction_date
        ]
        .copy()
    )


    scoring_data = (
        fcv_test_data.loc[
            fcv_test_data[
                "prediction_date"
            ]
            ==
            test_prediction_date
        ]
        .copy()
    )


    # ------------------------------------------------------------
    # Refit exact frozen Stage A model
    # ------------------------------------------------------------

    final_stage_a_model = clone(
        FINAL_FCV_STAGE_A_MODEL
    )


    final_stage_a_model.fit(
        development_data[
            FINAL_FCV_STAGE_A_FEATURES
        ],
        development_data[
            "future_active"
        ]
    )


    activity_probability = (
        final_stage_a_model
        .predict_proba(
            scoring_data[
                FINAL_FCV_STAGE_A_FEATURES
            ]
        )[:, 1]
    )


    # ------------------------------------------------------------
    # Frozen Stage B
    # ------------------------------------------------------------

    conditional_value = (
        0.50
        *
        scoring_data[
            "h180_revenue"
        ]
        .to_numpy()
    )


    expected_fcv90 = (
        activity_probability
        *
        conditional_value
    )


    output = (
        scoring_data[
            [
                "customer",
                "prediction_date",
                "prediction_period",
                "fcv90",
                "future_active",
                "h180_revenue",
            ]
        ]
        .copy()
    )


    output[
        "activity_probability"
    ] = activity_probability


    output[
        "conditional_value"
    ] = conditional_value


    output[
        "expected_fcv90"
    ] = expected_fcv90


    output[
        "development_episodes"
    ] = len(
        development_data
    )


    output[
        "latest_development_outcome_end"
    ] = (
        development_data[
            "outcome_end"
        ]
        .max()
    )


    final_fcv_test_prediction_frames.append(
        output
    )

In [ ]:
final_fcv_test_predictions = pd.concat(
    final_fcv_test_prediction_frames,
    ignore_index=True
)


print(
    f"Final Test predictions: "
    f"{len(final_fcv_test_predictions):,}"
)

In [ ]:
# ================================================================
# 55.4 — FINAL TEST REFIT AUDIT
# ================================================================

fcv_final_test_refit_audit = (
    final_fcv_test_predictions
    .groupby(
        "prediction_date",
        as_index=False
    )
    .agg(
        test_episodes=(
            "customer",
            "size"
        ),

        development_episodes=(
            "development_episodes",
            "first"
        ),

        latest_development_outcome_end=(
            "latest_development_outcome_end",
            "first"
        ),
    )
)


fcv_final_test_refit_audit[
    "label_timing_safe"
] = (
    fcv_final_test_refit_audit[
        "latest_development_outcome_end"
    ]
    <
    fcv_final_test_refit_audit[
        "prediction_date"
    ]
)


display(
    fcv_final_test_refit_audit
)


assert (
    fcv_final_test_refit_audit[
        "label_timing_safe"
    ]
    .all()
)

In [ ]:
# ================================================================
# 55.5 — OPEN FINAL TEST
# ================================================================

FCV_TEST_SET_OPENED = True


print(
    "FINAL FCV TEST POPULATION IS NOW OPEN."
)

print(
    "No further model modifications are permitted."
)

In [ ]:
# ================================================================
# 55.6 — FINAL TEST TARGET SUMMARY
# ================================================================

fcv_final_test_target_summary = pd.DataFrame(
    {
        "Metric": [
            "Episodes",
            "Future activity rate",
            "Mean FCV90",
            "Median FCV90",
            "Total FCV90",
        ],

        "Value": [
            len(
                final_fcv_test_predictions
            ),

            final_fcv_test_predictions[
                "future_active"
            ].mean(),

            final_fcv_test_predictions[
                "fcv90"
            ].mean(),

            final_fcv_test_predictions[
                "fcv90"
            ].median(),

            final_fcv_test_predictions[
                "fcv90"
            ].sum(),
        ],
    }
)


display(
    fcv_final_test_target_summary
)

In [ ]:
# ================================================================
# 55.7 — FINAL FCV TEST RESULTS
# ================================================================

final_fcv_test_results = (
    evaluate_complete_fcv_model(
        model_name=(
            "Frozen FCV90 Hurdle Model — Final Test"
        ),

        actual=
            final_fcv_test_predictions[
                "fcv90"
            ],

        predicted=
            final_fcv_test_predictions[
                "expected_fcv90"
            ],
    )
)


display(
    final_fcv_test_results
)

In [ ]:
# ================================================================
# 55.8 — FINAL STAGE A TEST PERFORMANCE
# ================================================================

final_fcv_stage_a_test_results = (
    evaluate_fcv_activity_model(
        model_name="Frozen Stage A Random Forest — Final Test",

        y_true=
            final_fcv_test_predictions[
                "future_active"
            ],

        probabilities=
            final_fcv_test_predictions[
                "activity_probability"
            ],
    )
)


display(
    final_fcv_stage_a_test_results
)

In [ ]:
# ================================================================
# 55.9 — FINAL TEST REVENUE CAPTURE
# ================================================================

final_fcv_test_capture_records = []


for fraction in [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
]:

    capture = (
        fcv_value_capture_at_fraction(
            final_fcv_test_predictions[
                "fcv90"
            ],

            final_fcv_test_predictions[
                "expected_fcv90"
            ],

            fraction
        )
    )


    final_fcv_test_capture_records.append(
        {
            "Customer Fraction":
                fraction,

            "Future Revenue Capture":
                capture,

            "Future Revenue Capture (%)":
                capture
                *
                100,
        }
    )


final_fcv_test_capture_curve = pd.DataFrame(
    final_fcv_test_capture_records
)


display(
    final_fcv_test_capture_curve
)

In [ ]:
# ================================================================
# 55.10 — FINAL FCV TEST DECILES
# ================================================================

final_fcv_test_deciles = (
    create_complete_fcv_deciles(
        actual=
            final_fcv_test_predictions[
                "fcv90"
            ],

        predicted=
            final_fcv_test_predictions[
                "expected_fcv90"
            ]
    )
)


display(
    final_fcv_test_deciles
)

In [ ]:
# ================================================================
# 55.11 — VALIDATION VS TEST
# ================================================================

fcv_validation_vs_test = pd.DataFrame(
    {
        "Metric": [
            "MAE",
            "RMSE",
            "R2",
            "Spearman",
            "Capture @ 10%",
            "Capture @ 20%",
            "Capture @ 30%",
            "Predicted / Actual Total",
        ],

        "Validation": [
            fcv_hurdle_run_rate_results[
                "MAE"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "RMSE"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "R2"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "Spearman"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "Capture @ 10%"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "Capture @ 20%"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "Capture @ 30%"
            ].iloc[0],

            fcv_hurdle_run_rate_results[
                "Predicted / Actual Total"
            ].iloc[0],
        ],

        "Test": [
            final_fcv_test_results[
                "MAE"
            ].iloc[0],

            final_fcv_test_results[
                "RMSE"
            ].iloc[0],

            final_fcv_test_results[
                "R2"
            ].iloc[0],

            final_fcv_test_results[
                "Spearman"
            ].iloc[0],

            final_fcv_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_fcv_test_results[
                "Capture @ 20%"
            ].iloc[0],

            final_fcv_test_results[
                "Capture @ 30%"
            ].iloc[0],

            final_fcv_test_results[
                "Predicted / Actual Total"
            ].iloc[0],
        ],
    }
)


fcv_validation_vs_test[
    "Absolute Change"
] = (
    fcv_validation_vs_test[
        "Test"
    ]
    -
    fcv_validation_vs_test[
        "Validation"
    ]
)


display(
    fcv_validation_vs_test
)

In [ ]:
# ================================================================
# 55.12 — STEP 55 COMPLETE
# ================================================================

assert FCV_TEST_SET_OPENED is True
assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is True
assert FCV_COMPLETE_MODEL_FROZEN is True


print(
    "✓ Frozen FCV architecture evaluated on final Test."
)

print(
    "✓ Test used once after model freeze."
)

print(
    "✓ No post-Test model modifications permitted."
)

### Step 55 Summary — Final Untouched FCV Test

The frozen Future Customer Value architecture was evaluated on the untouched July–August 2011 Test population.

The final architecture was:

> **Expected FCV90 = P(Future Active) × 50% × Prior 180-Day Revenue**

The Future Activity component used the frozen Random Forest classifier.

### Test Evaluation Was Temporally Safe

The Test was scored using rolling monthly refits.

For each Test prediction date, the Stage A classifier was trained only on historical episodes whose complete 90-day outcomes were known before the Test prediction date.

The label-availability audit passed for both July and August 2011.

### The Test Environment Was Economically Stronger

The Test population contained **5,534 episodes**.

Future activity was approximately:

> **58.2%**

Mean FCV90 was approximately:

> **798**

and total episode-level future revenue was approximately:

> **4.42 million**

The Test environment therefore represented materially stronger future customer economics than the preceding Validation period.

### Stage A Generalised Strongly

The frozen Future Activity Random Forest achieved approximately:

- **PR-AUC: 0.822**
- **ROC-AUC: 0.750**
- **Brier score: 0.203**
- **Lift@10%: 1.66×**
- **Lift@20%: 1.57×**

Stage A therefore demonstrated strong out-of-time discrimination.

### Complete FCV Ranking Remained Strong

The full FCV model achieved approximately:

- **Spearman: 0.550**
- **Capture@1%: 32.3%**
- **Capture@5%: 48.1%**
- **Capture@10%: 57.4%**
- **Capture@20%: 68.7%**
- **Capture@30%: 76.9%**

Revenue concentration remained strong despite the economic regime shift.

Compared with Validation, Spearman correlation increased from approximately **0.502 to 0.550**.

Capture@20% declined only modestly from approximately **71.4% to 68.7%**.

The architecture therefore generalised well as a customer-value ranking system.

### Absolute Revenue Calibration Deteriorated

The model predicted approximately:

> **2.62 million**

of Test FCV90 compared with approximately:

> **4.42 million actual**

The predicted-to-actual ratio was therefore approximately:

> **0.593**

This represents aggregate underprediction of approximately **40.7%**.

Absolute-value performance also weakened:

- MAE increased from approximately **349 to 526**;
- RMSE increased from approximately **1,210 to 2,514**;
- R² declined from approximately **0.712 to 0.620**.

The model therefore remained useful for economic ranking but did not maintain sufficient calibration for literal aggregate revenue forecasting.

## Final Decision

> **CONDITIONAL GO**

The FCV architecture is approved for:

- customer-value ranking;
- prioritisation;
- segmentation;
- CRM resource allocation;
- high-value customer identification;
- and controlled commercial decision support.

It is not approved in its current form for:

- accounting revenue forecasting;
- budget forecasting;
- exact individual future-revenue promises;
- or direct interpretation of Expected FCV90 as a calibrated monetary forecast.

The Test population has now been consumed.

No model parameter or calibration factor should be changed using Test outcomes.


## STEP 56 — OPERATING TIERS & MODEL CARD
We should close Notebook 5 properly.
### 56.1 — Create percentile-based FCV tiers
For operational use, use relative ranks, not hard currency thresholds.

This is appropriate because the Test showed that ranking generalises better than absolute monetary calibration.

In [ ]:
# ================================================================
# 56.1 — FCV OPERATING TIERS
# ================================================================

final_fcv_test_predictions[
    "fcv_percentile"
] = (
    final_fcv_test_predictions[
        "expected_fcv90"
    ]
    .rank(
        method="average",
        pct=True
    )
)


final_fcv_test_predictions[
    "fcv_tier"
] = pd.cut(
    final_fcv_test_predictions[
        "fcv_percentile"
    ],
    bins=[
        0.00,
        0.50,
        0.80,
        0.90,
        0.95,
        1.00,
    ],
    labels=[
        "Core",
        "Growth",
        "High Value",
        "Priority",
        "Premier",
    ],
    include_lowest=True
)


fcv_tier_summary = (
    final_fcv_test_predictions
    .groupby(
        "fcv_tier",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        average_expected_fcv90=(
            "expected_fcv90",
            "mean"
        ),

        average_actual_fcv90=(
            "fcv90",
            "mean"
        ),

        total_actual_fcv90=(
            "fcv90",
            "sum"
        ),

        future_activity_rate=(
            "future_active",
            "mean"
        ),
    )
)


fcv_tier_summary[
    "share_of_actual_future_revenue"
] = (
    fcv_tier_summary[
        "total_actual_fcv90"
    ]
    /
    final_fcv_test_predictions[
        "fcv90"
    ].sum()
)


display(
    fcv_tier_summary
)

In [ ]:
# ================================================================
# 56.2 — FINAL FCV MODEL CARD
# ================================================================

final_fcv_model_card = pd.DataFrame(
    {
        "Item": [
            "Model objective",
            "Target",
            "Prediction horizon",
            "Population",
            "Stage A",
            "Stage B",
            "Final formula",
            "Final decision",
            "Approved use",
            "Not approved for",
            "Stage A Test PR-AUC",
            "Stage A Test ROC-AUC",
            "Complete Test Spearman",
            "Test Capture@10%",
            "Test Capture@20%",
            "Test Capture@30%",
            "Test predicted/actual revenue",
            "Primary limitation",
            "Test status",
        ],

        "Value": [
            "Rank customers by expected near-term economic value",
            "Future 90-Day Customer Value (FCV90)",
            "90 days",
            "Identified customers with purchase in prior 180 days",
            "Random Forest Future Activity Classifier",
            "50% × prior 180-day revenue",
            "P(Future Active) × 50% × prior 180-day revenue",
            "CONDITIONAL GO",
            "Ranking, segmentation, prioritisation and CRM decision support",
            "Accounting forecast, budget forecast or exact revenue promise",
            0.822148,
            0.749591,
            0.550464,
            0.573750,
            0.686870,
            0.768946,
            0.593481,
            "Absolute monetary calibration drift under changing economic regimes",
            "CONSUMED",
        ],
    }
)


display(
    final_fcv_model_card
)

In [ ]:
# ================================================================
# 56.3 — FINAL GOVERNANCE STATUS
# ================================================================

FCV_TEST_SET_CONSUMED = True
NOTEBOOK_5_COMPLETE = True

FINAL_FCV_DECISION = (
    "CONDITIONAL GO"
)


assert FCV_TEST_SET_OPENED is True
assert FCV_TEST_SET_CONSUMED is True
assert FCV_STAGE_A_MODEL_FROZEN is True
assert FCV_STAGE_B_MODEL_FROZEN is True
assert FCV_COMPLETE_MODEL_FROZEN is True


print(
    f"Final FCV decision: "
    f"{FINAL_FCV_DECISION}"
)

print(
    "✓ Final Test consumed."
)

print(
    "✓ No post-Test tuning permitted."
)

print(
    "✓ Ranking use approved."
)

print(
    "✓ Literal aggregate revenue forecasting not approved."
)

print(
    "✓ Notebook 5 complete."
)

# Final Notebook Conclusion

## Future Customer Value

This notebook developed and validated a fixed-horizon customer-value framework designed to estimate near-term customer economic importance.

The target was:

> **FCV90 — Future 90-Day Customer Value**

FCV90 represents qualifying merchandise revenue generated by an identified customer during the 90 days following the prediction point.

It is not a true lifetime-value measure.

## Two-Part Economic Architecture

Target diagnostics showed that approximately half of historical prediction episodes generated zero future revenue while positive future-revenue outcomes were strongly right-skewed.

A two-part hurdle framework was therefore developed.

### Stage A — Future Activity

The first stage predicts:

> **P(FCV90 > 0)**

The frozen Stage A model is a Random Forest classifier.

### Stage B — Conditional Economic Scale

The second stage estimates the economic scale associated with an active customer using:

> **50% × Prior 180-Day Revenue**

### Final FCV Score

The final frozen architecture is:

> **Expected FCV90 = P(Future Active) × 50% × Prior 180-Day Revenue**

## Final Test Performance

The final Test contained **5,534 customer prediction episodes**.

Stage A achieved approximately:

- **PR-AUC: 0.822**
- **ROC-AUC: 0.750**
- **Lift@10%: 1.66×**
- **Lift@20%: 1.57×**

The complete FCV system achieved approximately:

- **Spearman: 0.550**
- **Capture@10%: 57.4%**
- **Capture@20%: 68.7%**
- **Capture@30%: 76.9%**

The model therefore successfully concentrated future customer revenue into a relatively small proportion of customers.

## Ranking Generalised Better Than Absolute Revenue Calibration

Customer ranking remained strong out of time.

Spearman correlation improved from approximately **0.502 on Validation to 0.550 on Test**.

Revenue capture remained relatively stable.

However, absolute revenue calibration weakened materially.

The model predicted approximately **59.3% of actual Test future revenue**, corresponding to aggregate underprediction of approximately **40.7%**.

This demonstrates that the architecture is substantially more reliable for:

> **relative customer-value ranking**

than for:

> **literal monetary forecasting**

## Final Decision

> **CONDITIONAL GO**

The FCV system is approved for:

- customer ranking;
- value segmentation;
- CRM prioritisation;
- high-value customer identification;
- customer-service prioritisation;
- and controlled commercial decision support.

The model is not approved in its current form for:

- accounting revenue forecasts;
- financial budgets;
- exact individual future-revenue expectations;
- or interpreting Expected FCV90 as guaranteed recoverable or future revenue.

Future recalibration should be performed only with genuinely new data and must not reuse the consumed Test population for model tuning.

---

**Notebook 5 Status: COMPLETE**

In [ ]:
# ================================================================
# EXPORT AUDIT — FUTURE CUSTOMER VALUE NOTEBOOK
# ================================================================

fcv_export_candidates = []


search_terms = [
    "fcv",
    "future",
    "value",
    "expected",
    "prob",
    "score",
    "prediction",
    "pred",
]


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns = list(
        obj.columns
    )


    columns_lower = [
        str(column).lower()
        for column in columns
    ]


    has_customer = (
        "customer"
        in columns_lower
    )


    matching_columns = [
        column
        for column in columns
        if any(
            term
            in str(column).lower()
            for term in search_terms
        )
    ]


    if (
        has_customer
        and
        matching_columns
    ):

        fcv_export_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Matching Columns":
                    matching_columns,

                "All Columns":
                    columns,
            }
        )


fcv_export_candidates = pd.DataFrame(
    fcv_export_candidates
)


display(
    fcv_export_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 2 — FINAL FCV TEST PREDICTIONS
# ================================================================

print(
    "final_fcv_test_predictions shape:",
    final_fcv_test_predictions.shape
)

print(
    "\nColumns:"
)

for column in final_fcv_test_predictions.columns:

    print(
        " -",
        column
    )


display(
    final_fcv_test_predictions.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT AUDIT 3 — FCV OUTPUT STRUCTURE
# ================================================================

print(
    "output shape:",
    output.shape
)

print(
    "\nColumns:"
)

for column in output.columns:

    print(
        " -",
        column
    )


display(
    output.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT — FUTURE CUSTOMER VALUE MODEL OUTPUT
# ================================================================

from pathlib import Path

MODEL_OUTPUT_DIR = Path("model_outputs")

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Construct clean historical scoring interface
# ------------------------------------------------

future_customer_value_scores = (
    final_fcv_test_predictions[
        [
            "customer",
            "prediction_date",
            "prediction_period",
            "h180_revenue",
            "activity_probability",
            "expected_fcv90",
            "fcv_percentile",
            "fcv_tier",
            "development_episodes",
            "latest_development_outcome_end",
        ]
    ]
    .copy()
)


# ------------------------------------------------
# Canonical Notebook 8 naming
# ------------------------------------------------

future_customer_value_scores = (
    future_customer_value_scores
    .rename(
        columns={
            "h180_revenue":
                "fcv_history_180d_revenue",

            "activity_probability":
                "future_active_probability",

            "expected_fcv90":
                "future_customer_value_90d",
        }
    )
)


# ------------------------------------------------
# Standardise customer identifier
# ------------------------------------------------

future_customer_value_scores[
    "customer"
] = (
    pd.to_numeric(
        future_customer_value_scores[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


future_customer_value_scores[
    "prediction_date"
] = pd.to_datetime(
    future_customer_value_scores[
        "prediction_date"
    ]
)


future_customer_value_scores[
    "latest_development_outcome_end"
] = pd.to_datetime(
    future_customer_value_scores[
        "latest_development_outcome_end"
    ]
)


# ------------------------------------------------
# Explicit model provenance
# ------------------------------------------------

future_customer_value_scores[
    "fcv_model_scope"
] = (
    "Expected positive merchandise revenue "
    "over the following 90 days"
)


future_customer_value_scores[
    "fcv_value_interpretation"
] = (
    "Relative customer-value prioritisation signal; "
    "not calibrated lifetime value or accounting forecast"
)


future_customer_value_scores[
    "fcv_score_provenance"
] = (
    "historical held-out Test customer-month scoring"
)


# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert (
    future_customer_value_scores[
        "customer"
    ]
    .notna()
    .all()
)


assert (
    future_customer_value_scores[
        "prediction_date"
    ]
    .notna()
    .all()
)


assert (
    future_customer_value_scores[
        "future_active_probability"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    future_customer_value_scores[
        "fcv_percentile"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    future_customer_value_scores[
        "future_customer_value_90d"
    ]
    .ge(
        0
    )
    .all()
)


assert (
    future_customer_value_scores[
        [
            "customer",
            "prediction_date",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
), (
    "Expected one FCV score per customer "
    "per prediction date."
)


# ------------------------------------------------
# Save persistent output
# ------------------------------------------------

fcv_output_path = (
    MODEL_OUTPUT_DIR
    /
    "future_customer_value_scores.parquet"
)


future_customer_value_scores.to_parquet(
    fcv_output_path,
    index=False
)


# ------------------------------------------------
# Export audit
# ------------------------------------------------

print(
    "✓ Future Customer Value scores exported."
)

print(
    fcv_output_path.resolve()
)

print(
    f"Rows: "
    f"{len(future_customer_value_scores):,}"
)

print(
    f"Customers: "
    f"{future_customer_value_scores['customer'].nunique():,}"
)

print(
    f"Prediction date range: "
    f"{future_customer_value_scores['prediction_date'].min()} "
    f"to "
    f"{future_customer_value_scores['prediction_date'].max()}"
)

print(
    f"Future-active probability range: "
    f"{future_customer_value_scores['future_active_probability'].min():.4f} "
    f"to "
    f"{future_customer_value_scores['future_active_probability'].max():.4f}"
)

print(
    f"Expected FCV90 range: "
    f"{future_customer_value_scores['future_customer_value_90d'].min():,.2f} "
    f"to "
    f"{future_customer_value_scores['future_customer_value_90d'].max():,.2f}"
)


display(
    future_customer_value_scores.head(
        10
    )
)